# YOLO Colonies + MLflow (736, mask_ratio=2)

Pipeline in this notebook:
1. Detection-based crop of Petri dish + resize to 1024x1024 with label remap
2. Same offline augmentations as before (leaky split strategy by request)
3. Train YOLO-seg models with MLflow (`imgsz=736`, `mask_ratio=2`)


In [ ]:
from pathlib import Path
import random
import shutil
import zlib

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -------- Source (before Petri crop) --------
RAW_ROOT = Path("Новая папка")
RAW_IMG_DIR = RAW_ROOT / "images" / "train"
RAW_LBL_DIR = RAW_ROOT / "labels" / "train"

# -------- Petri detector --------
CROP_WEIGHTS = Path("runs/detect/runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt")
CROP_CONF = 0.25
CROP_IOU = 0.5

# -------- Cropped dataset (target 1024x1024) --------
DATASET_ROOT = Path("cropped_736")
SRC_IMG_DIR = DATASET_ROOT / "images" / "train"
SRC_LBL_DIR = DATASET_ROOT / "labels" / "train"
TARGET_W = 736
TARGET_H = 736
OVERWRITE_CROPPED = True

# -------- Offline-aug workspace --------
WORK_ROOT = DATASET_ROOT.parent / "cropped_736_aug_leaky"
POOL_IMG_DIR = WORK_ROOT / "_pool" / "images"
POOL_LBL_DIR = WORK_ROOT / "_pool" / "labels"
SPLIT_ROOT = WORK_ROOT / "dataset"

for req in [RAW_IMG_DIR, RAW_LBL_DIR]:
    if not req.exists():
        raise FileNotFoundError(f"Required path not found: {req}")
if not CROP_WEIGHTS.exists():
    raise FileNotFoundError(f"Crop model not found: {CROP_WEIGHTS}")

print(f"RAW_ROOT: {RAW_ROOT.resolve()}")
print(f"DATASET_ROOT (cropped): {DATASET_ROOT.resolve()}")
print(f"WORK_ROOT: {WORK_ROOT.resolve()}")
print(f"CROP_WEIGHTS: {CROP_WEIGHTS.resolve()}")




In [ ]:
# Detection-based crop + label remap -> cropped_736

def load_yolo_seg_crop(txt_path: Path):
    anns = []
    if not txt_path.exists():
        return anns
    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    text = text.replace("\r", "\n").replace("\n", "\n")
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.replace(",", " ").split()
        if len(parts) < 7:
            continue
        cls = parts[0]
        coords = []
        for tok in parts[1:]:
            try:
                coords.append(float(tok))
            except ValueError:
                pass
        if len(coords) < 6:
            continue
        if len(coords) % 2 != 0:
            coords = coords[:-1]
        if len(coords) < 6:
            continue
        pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
        anns.append((cls, pts))
    return anns


def transform_seg_crop(coords_norm, img_w, img_h, x1, y1, x2, y2):
    pts = coords_norm.copy()
    pts[:, 0] *= img_w
    pts[:, 1] *= img_h

    min_x, max_x = float(pts[:, 0].min()), float(pts[:, 0].max())
    min_y, max_y = float(pts[:, 1].min()), float(pts[:, 1].max())
    if max_x <= x1 or min_x >= x2 or max_y <= y1 or min_y >= y2:
        return None

    pts[:, 0] = np.clip(pts[:, 0], x1, x2 - 1e-6)
    pts[:, 1] = np.clip(pts[:, 1], y1, y2 - 1e-6)

    crop_w = max(1e-6, float(x2 - x1))
    crop_h = max(1e-6, float(y2 - y1))
    pts[:, 0] = (pts[:, 0] - x1) / crop_w
    pts[:, 1] = (pts[:, 1] - y1) / crop_h
    pts = np.clip(pts, 0.0, 1.0)

    if (pts[:, 0].max() - pts[:, 0].min()) < 1e-4:
        return None
    if (pts[:, 1].max() - pts[:, 1].min()) < 1e-4:
        return None
    return pts


def write_yolo_seg_crop(txt_path: Path, anns):
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for cls, pts in anns:
        pts = np.clip(pts, 0.0, 1.0)
        flat = pts.reshape(-1)
        lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in flat))
    txt_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


if OVERWRITE_CROPPED and DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)

SRC_IMG_DIR.mkdir(parents=True, exist_ok=True)
SRC_LBL_DIR.mkdir(parents=True, exist_ok=True)

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
images = sorted([p for p in RAW_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in image_exts])
if not images:
    raise RuntimeError(f"No images found in: {RAW_IMG_DIR}")

crop_model = YOLO(str(CROP_WEIGHTS))

saved = 0
fallback_full = 0
dropped_masks = 0

for i, img_path in enumerate(images, start=1):
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img is None:
        continue

    h, w = img.shape[:2]
    pred = crop_model.predict(source=str(img_path), conf=CROP_CONF, iou=CROP_IOU, verbose=False)[0]

    if pred.boxes is None or len(pred.boxes) == 0:
        x1, y1, x2, y2 = 0, 0, w, h
        fallback_full += 1
    else:
        confs = pred.boxes.conf.detach().cpu().numpy()
        boxes = pred.boxes.xyxy.detach().cpu().numpy()
        best_idx = int(np.argmax(confs))
        bx1, by1, bx2, by2 = boxes[best_idx]
        x1 = max(0, int(np.floor(bx1)))
        y1 = max(0, int(np.floor(by1)))
        x2 = min(w, int(np.ceil(bx2)))
        y2 = min(h, int(np.ceil(by2)))
        if x2 <= x1 or y2 <= y1:
            x1, y1, x2, y2 = 0, 0, w, h
            fallback_full += 1

    crop = img[y1:y2, x1:x2]
    interp = cv2.INTER_AREA if crop.shape[1] >= TARGET_W and crop.shape[0] >= TARGET_H else cv2.INTER_LINEAR
    resized = cv2.resize(crop, (TARGET_W, TARGET_H), interpolation=interp)

    out_img = SRC_IMG_DIR / img_path.name
    cv2.imwrite(str(out_img), resized)

    in_lbl = RAW_LBL_DIR / f"{img_path.stem}.txt"
    anns = load_yolo_seg_crop(in_lbl)

    out_anns = []
    for cls, coords in anns:
        t = transform_seg_crop(coords, w, h, x1, y1, x2, y2)
        if t is None:
            dropped_masks += 1
            continue
        out_anns.append((cls, t))

    out_lbl = SRC_LBL_DIR / f"{img_path.stem}.txt"
    write_yolo_seg_crop(out_lbl, out_anns)

    saved += 1
    if i % 25 == 0 or i == len(images):
        print(f"Processed {i}/{len(images)}")

print(f"Saved cropped images: {saved}")
print(f"Fallback full-image crops: {fallback_full}")
print(f"Dropped polygons after crop: {dropped_masks}")
print(f"Cropped images dir: {SRC_IMG_DIR.resolve()}")



In [ ]:
# Offline augmentation + split (same scheme as previous notebook)
OVERWRITE_WORK_ROOT = True

GEOM_MODES = ["none", "hflip", "vflip", "hvflip"]
PHOTO_MODES = ["none", "clahe", "gamma", "hsv", "blur", "noise"]
PHOTO_REPEATS = 3

VARIANTS = []
for geom in GEOM_MODES:
    for photo in PHOTO_MODES:
        if photo == "none":
            name = "orig" if geom == "none" else f"{geom}_orig"
            VARIANTS.append({"name": name, "geom": geom, "photo": photo})
        else:
            for rep in range(1, PHOTO_REPEATS + 1):
                VARIANTS.append({"name": f"{geom}_{photo}_r{rep}", "geom": geom, "photo": photo})

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
OUTPUT_IMAGE_EXT = ".jpg"


def read_yolo_seg(label_path: Path):
    anns = []
    if not label_path.exists():
        return anns

    text = label_path.read_text(encoding="utf-8", errors="ignore")
    text = text.replace("\r", "\n").replace("\n", "\n")

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.replace(",", " ").split()
        if len(parts) < 7:
            continue

        cls = parts[0]
        coords = []
        for token in parts[1:]:
            try:
                coords.append(float(token))
            except ValueError:
                pass

        if len(coords) < 6:
            continue
        if len(coords) % 2 != 0:
            coords = coords[:-1]
        if len(coords) < 6:
            continue

        pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
        anns.append((cls, pts))
    return anns


def write_yolo_seg(label_path: Path, anns):
    label_path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for cls, pts in anns:
        pts = np.clip(pts, 0.0, 1.0)
        flat = pts.reshape(-1)
        lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in flat))
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


def transform_points(pts, geom):
    out = pts.copy()
    if geom in {"hflip", "hvflip"}:
        out[:, 0] = 1.0 - out[:, 0]
    if geom in {"vflip", "hvflip"}:
        out[:, 1] = 1.0 - out[:, 1]
    return np.clip(out, 0.0, 1.0)


def apply_geom(image, anns, geom):
    if geom == "none":
        return image, anns
    if geom == "hflip":
        out_img = cv2.flip(image, 1)
    elif geom == "vflip":
        out_img = cv2.flip(image, 0)
    elif geom == "hvflip":
        out_img = cv2.flip(image, -1)
    else:
        raise ValueError(f"Unknown geom transform: {geom}")

    out_anns = [(cls, transform_points(pts, geom)) for cls, pts in anns]
    return out_img, out_anns


def apply_photo(image, photo, rng):
    if photo == "none":
        return image

    img = image.copy()

    if photo == "clahe":
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clip = float(rng.uniform(2.0, 4.0))
        clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
        l2 = clahe.apply(l)
        return cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)

    if photo == "gamma":
        gamma = float(rng.uniform(0.7, 1.5))
        inv = 1.0 / gamma
        table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)], dtype=np.uint8)
        return cv2.LUT(img, table)

    if photo == "hsv":
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        h_gain = float(rng.uniform(-8, 8))
        s_gain = float(rng.uniform(0.75, 1.35))
        v_gain = float(rng.uniform(0.75, 1.35))
        hsv[..., 0] = (hsv[..., 0] + h_gain) % 180
        hsv[..., 1] = np.clip(hsv[..., 1] * s_gain, 0, 255)
        hsv[..., 2] = np.clip(hsv[..., 2] * v_gain, 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    if photo == "blur":
        k = int(rng.choice([3, 5]))
        sigma = float(rng.uniform(0.3, 1.2))
        return cv2.GaussianBlur(img, (k, k), sigmaX=sigma)

    if photo == "noise":
        sigma = float(rng.uniform(4.0, 14.0))
        noise = rng.normal(0, sigma, size=img.shape).astype(np.float32)
        out = np.clip(img.astype(np.float32) + noise, 0, 255)
        return out.astype(np.uint8)

    raise ValueError(f"Unknown photo transform: {photo}")


if OVERWRITE_WORK_ROOT and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

POOL_IMG_DIR.mkdir(parents=True, exist_ok=True)
POOL_LBL_DIR.mkdir(parents=True, exist_ok=True)

src_images = sorted([p for p in SRC_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
if not src_images:
    raise RuntimeError(f"No source images found in: {SRC_IMG_DIR}")

written = 0
for i, img_path in enumerate(src_images, start=1):
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img is None:
        continue

    in_label = SRC_LBL_DIR / f"{img_path.stem}.txt"
    anns = read_yolo_seg(in_label)

    for var in VARIANTS:
        out_stem = f"{img_path.stem}__{var['name']}"
        seed_local = SEED + zlib.crc32(out_stem.encode("utf-8"))
        rng = np.random.default_rng(seed_local)

        g_img, g_anns = apply_geom(img, anns, var["geom"])
        p_img = apply_photo(g_img, var["photo"], rng)

        out_img = POOL_IMG_DIR / f"{out_stem}{OUTPUT_IMAGE_EXT}"
        out_lbl = POOL_LBL_DIR / f"{out_stem}.txt"

        cv2.imwrite(str(out_img), p_img)
        write_yolo_seg(out_lbl, g_anns)
        written += 1

    if i % 10 == 0 or i == len(src_images):
        print(f"Augmented {i}/{len(src_images)} source images")

print(f"Source images: {len(src_images)}")
print(f"Variants per image: {len(VARIANTS)}")
print(f"Pool samples written: {written}")

# Split augmented pool into train/val/test AFTER augmentation (leaky by design)
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
SPLIT_SEED = 42

if abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) > 1e-9:
    raise ValueError("Split ratios must sum to 1.0")

pool_images = sorted([p for p in POOL_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
pool_stems = [p.stem for p in pool_images]

rng = random.Random(SPLIT_SEED)
rng.shuffle(pool_stems)

n = len(pool_stems)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)
n_test = n - n_train - n_val

splits = {
    "train": pool_stems[:n_train],
    "val": pool_stems[n_train:n_train + n_val],
    "test": pool_stems[n_train + n_val:],
}

for split_name in ["train", "val", "test"]:
    (SPLIT_ROOT / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (SPLIT_ROOT / "labels" / split_name).mkdir(parents=True, exist_ok=True)

for split_name, stems in splits.items():
    img_dst = SPLIT_ROOT / "images" / split_name
    lbl_dst = SPLIT_ROOT / "labels" / split_name

    for stem in stems:
        src_img = POOL_IMG_DIR / f"{stem}{OUTPUT_IMAGE_EXT}"
        src_lbl = POOL_LBL_DIR / f"{stem}.txt"

        shutil.copy2(src_img, img_dst / src_img.name)
        if src_lbl.exists():
            shutil.copy2(src_lbl, lbl_dst / src_lbl.name)
        else:
            (lbl_dst / f"{stem}.txt").write_text("", encoding="utf-8")

print(f"Total: {n}")
print(f"Train: {len(splits['train'])}")
print(f"Val:   {len(splits['val'])}")
print(f"Test:  {len(splits['test'])}")

# Write data config for YOLO
DATA_YAML = SPLIT_ROOT / "data.yaml"
train_dir = (SPLIT_ROOT / "images" / "train").resolve().as_posix()
val_dir = (SPLIT_ROOT / "images" / "val").resolve().as_posix()
test_dir = (SPLIT_ROOT / "images" / "test").resolve().as_posix()

yaml_text = "\n".join([
    f'train: "{train_dir}"',
    f'val: "{val_dir}"',
    f'test: "{test_dir}"',
    "nc: 1",
    "names: [colony]",
]) + "\n"

DATA_YAML.write_text(yaml_text, encoding="utf-8")
print("DATA_YAML:", DATA_YAML.resolve())
print(yaml_text)



# Optional: simplify train polygons only (keep val/test unchanged)
SIMPLIFY_TRAIN_LABELS = True
SIMPLIFY_EPS_FRAC = 0.0035      # epsilon = frac * polygon_perimeter
SIMPLIFY_MIN_VERTICES = 12      # keep round colonies reasonably smooth
SIMPLIFY_MAX_AREA_DELTA = 0.10  # reject simplification if area shifts >10%
SIMPLIFY_BACKUP_ORIGINAL = True


def simplify_polygon_safe(pts_norm, img_w, img_h, eps_frac, min_vertices, max_area_delta):
    """Soft Douglas-Peucker simplification with safety checks."""
    pts = np.asarray(pts_norm, dtype=np.float32)
    if pts.ndim != 2 or pts.shape[1] != 2 or pts.shape[0] < 3:
        return pts

    cnt = np.empty_like(pts)
    cnt[:, 0] = np.clip(pts[:, 0] * (img_w - 1), 0, img_w - 1)
    cnt[:, 1] = np.clip(pts[:, 1] * (img_h - 1), 0, img_h - 1)
    cnt_cv = cnt.reshape(-1, 1, 2).astype(np.float32)

    perimeter = float(cv2.arcLength(cnt_cv, True))
    if not np.isfinite(perimeter) or perimeter <= 1e-6:
        return pts

    orig_area = float(abs(cv2.contourArea(cnt_cv)))
    if not np.isfinite(orig_area) or orig_area <= 1e-6:
        return pts

    eps = max(0.5, float(eps_frac) * perimeter)
    approx_cv = cv2.approxPolyDP(cnt_cv, epsilon=eps, closed=True)

    if approx_cv is None:
        return pts

    approx = approx_cv.reshape(-1, 2)
    if approx.shape[0] < max(3, int(min_vertices)):
        return pts

    new_area = float(abs(cv2.contourArea(approx_cv)))
    if not np.isfinite(new_area) or new_area <= 1e-6:
        return pts

    rel_area_delta = abs(new_area - orig_area) / max(orig_area, 1e-6)
    if rel_area_delta > float(max_area_delta):
        return pts

    out = np.empty_like(approx, dtype=np.float32)
    out[:, 0] = np.clip(approx[:, 0] / max(1, (img_w - 1)), 0.0, 1.0)
    out[:, 1] = np.clip(approx[:, 1] / max(1, (img_h - 1)), 0.0, 1.0)
    return out


if SIMPLIFY_TRAIN_LABELS:
    split_name = "train"
    split_img_dir = SPLIT_ROOT / "images" / split_name
    split_lbl_dir = SPLIT_ROOT / "labels" / split_name

    if SIMPLIFY_BACKUP_ORIGINAL:
        backup_dir = SPLIT_ROOT / "labels" / f"{split_name}_orig_before_simplify"
        if backup_dir.exists():
            shutil.rmtree(backup_dir)
        shutil.copytree(split_lbl_dir, backup_dir)
        print(f"Backup labels: {backup_dir.resolve()}")

    img_map = {
        p.stem: p
        for p in split_img_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    }

    files_processed = 0
    missing_images = 0
    polygons_total = 0
    polygons_simplified = 0

    for lbl_path in sorted(split_lbl_dir.glob("*.txt")):
        anns = read_yolo_seg(lbl_path)
        if not anns:
            continue

        img_path = img_map.get(lbl_path.stem)
        if img_path is None:
            missing_images += 1
            continue

        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img is None:
            missing_images += 1
            continue

        h, w = img.shape[:2]
        out_anns = []

        for cls, pts in anns:
            polygons_total += 1
            simp = simplify_polygon_safe(
                pts_norm=pts,
                img_w=w,
                img_h=h,
                eps_frac=SIMPLIFY_EPS_FRAC,
                min_vertices=SIMPLIFY_MIN_VERTICES,
                max_area_delta=SIMPLIFY_MAX_AREA_DELTA,
            )

            changed = (simp.shape[0] != pts.shape[0]) or (not np.allclose(simp, pts, atol=1e-6))
            if changed:
                polygons_simplified += 1

            out_anns.append((cls, simp))

        write_yolo_seg(lbl_path, out_anns)
        files_processed += 1

    ratio = (100.0 * polygons_simplified / max(1, polygons_total))
    print(f"Simplify split: {split_name}")
    print(f"Files processed: {files_processed}")
    print(f"Missing images: {missing_images}")
    print(f"Polygons simplified: {polygons_simplified}/{polygons_total} ({ratio:.2f}%)")
else:
    print("Train polygon simplification is disabled.")


# YOLO Segmentation + MLflow (Multi-model)

This notebook is focused only on MLflow-managed training for:
- `yolo26n-seg.pt`
- `yolo26s-seg.pt`
- `yolo26m-seg.pt`
- `yolo26l-seg.pt`
- `yolo26x-seg.pt`

It expects a prepared dataset YAML (`data.yaml`) from your main pipeline.


In [ ]:
from pathlib import Path
import json
import torch

import numpy as np
import pandas as pd
import mlflow
from ultralytics import YOLO

# Set your data YAML path explicitly if needed
if "DATA_YAML" not in globals() or DATA_YAML is None:
    DATA_YAML = Path('cropped_736_aug_leaky/dataset/data.yaml')
else:
    DATA_YAML = Path(DATA_YAML)

# Fallback locations
if not DATA_YAML.exists():
    candidates = [
        Path('C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml'),
    ]
    for c in candidates:
        if c.exists():
            DATA_YAML = c
            break

if not DATA_YAML.exists():
    # Last-resort search (first match)
    found = sorted(Path('.').rglob('cropped_736_aug_leaky/dataset/data.yaml'))
    if found:
        DATA_YAML = found[0]

if not DATA_YAML.exists():
    raise FileNotFoundError('data.yaml not found. Prepare dataset first and set DATA_YAML manually.')

print('DATA_YAML:', DATA_YAML.resolve())



In [ ]:
# MLflow: train YOLO-seg models with VRAM-safe defaults (n/s by default)
from pathlib import Path
import json
import os
import sys
import threading
import gc

# Reduce CUDA memory fragmentation on long notebook sessions
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:64")
if "torch" in sys.modules:
    print("[WARN] torch already imported before CUDA alloc config; restart kernel for full effect.")

import cv2
import mlflow
from mlflow.exceptions import MlflowException
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO, settings as yolo_settings

# Ensure UTF-8 output to avoid MLflow unicode print errors on Windows cp1251 consoles
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

# If you have MLflow server, set e.g. "http://127.0.0.1:5000".
# If None, local file-backed MLflow store is used.
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"

MLFLOW_EXPERIMENT = "colony_yolo_seg_736"
LOCAL_MLFLOW_DIR = Path("mlruns_yolo_seg")

# Default set for 12GB VRAM with imgsz>=736 and mask_ratio=2
MODELS_TO_TRAIN = [
    "yolo26n-seg.pt",
    "yolo26s-seg.pt",
]
# To try heavier models manually (still may OOM on 12GB with mask_ratio=2):
# MODELS_TO_TRAIN = ["yolo26m-seg.pt", "yolo26l-seg.pt", "yolo26x-seg.pt"]

MLFLOW_PROJECT = "runs/colony_seg_mlflow_736"
RUN_SUFFIX = "cropped736_offline_aug_mr2"

# Training setup
TRAIN_EPOCHS = 50
TRAIN_BATCH = 1
TRAIN_BATCH_BY_MODEL = {
    "yolo26n-seg.pt": 1,
    "yolo26s-seg.pt": 1,
    "yolo26m-seg.pt": 1,
    "yolo26l-seg.pt": 1,
    "yolo26x-seg.pt": 1,
}
TRAIN_WORKERS = 0
TRAIN_OVERLAP_MASK = False
TRAIN_IMGSZ = 736
TRAIN_PATIENCE = 80
TRAIN_DEVICE = 0  # use first GPU explicitly
TRAIN_OPTIMIZER = "SGD"   # lower VRAM than AdamW
TRAIN_AMP = True           # mixed precision saves VRAM
TRAIN_CACHE = False
STOP_ON_ERROR = False

# Custom metric inference setup (for Dice + counting quality)
PRED_CONF = 0.25
PRED_IOU = 0.7
PRED_MAX_DET = 300
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
LIVE_LOG_POLL_SEC = 2.0


def to_float(v):
    try:
        return float(v)
    except Exception:
        return float("nan")


def summarize_section(section, prefix):
    if section is None:
        return {
            f"precision_{prefix}": float("nan"),
            f"recall_{prefix}": float("nan"),
            f"mAP50_{prefix}": float("nan"),
            f"mAP50_95_{prefix}": float("nan"),
        }
    return {
        f"precision_{prefix}": to_float(getattr(section, "mp", float("nan"))),
        f"recall_{prefix}": to_float(getattr(section, "mr", float("nan"))),
        f"mAP50_{prefix}": to_float(getattr(section, "map50", float("nan"))),
        f"mAP50_95_{prefix}": to_float(getattr(section, "map", float("nan"))),
    }


def safe_metric_name(name: str) -> str:
    s = str(name).strip().replace(" ", "_")
    for bad in ["(", ")", "/", "\\", ":", ",", "|", "-", "."]:
        s = s.replace(bad, "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")


def clear_cuda_memory(note: str = ""):
    gc.collect()
    suffix = f" ({note})" if note else ""

    if not torch.cuda.is_available():
        print(f"CUDA not available{suffix}")
        return

    try:
        if torch.cuda.is_initialized():
            try:
                torch.cuda.synchronize()
            except Exception:
                pass

            try:
                torch.cuda.empty_cache()
            except RuntimeError as e:
                print(f"[WARN] empty_cache failed{suffix}: {e}")

            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass

        print(f"CUDA cache clear attempted{suffix}")
    except Exception as e:
        print(f"[WARN] CUDA cleanup failed{suffix}: {e}")
        print("[INFO] If this follows OOM, do Kernel Restart to reset CUDA context.")


def maybe_log_artifact(path: Path, artifact_path: str):
    if path.exists():
        mlflow.log_artifact(str(path), artifact_path=artifact_path)


def resolve_test_split_dirs(data_yaml_path: Path):
    data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8")) or {}
    test_raw = data.get("test")
    if not test_raw:
        raise KeyError("`test` path is missing in data.yaml")

    test_img_dir = Path(str(test_raw).strip().strip('"').strip("'"))
    if not test_img_dir.is_absolute():
        test_img_dir = (data_yaml_path.parent / test_img_dir).resolve()

    split_name = test_img_dir.name
    label_candidates = [data_yaml_path.parent / "labels" / split_name]

    if test_img_dir.parent.name == "images":
        label_candidates.append(test_img_dir.parent.parent / "labels" / split_name)

    replaced = Path(
        str(test_img_dir)
        .replace("\\images\\", "\\labels\\")
        .replace("/images/", "/labels/")
    )
    label_candidates.append(replaced)

    for cand in label_candidates:
        if cand.exists():
            return test_img_dir, cand

    return test_img_dir, label_candidates[0]


def read_yolo_seg_polygons(label_path: Path):
    if not label_path.exists():
        return []

    txt = label_path.read_text(encoding="utf-8", errors="ignore")
    if not txt.strip():
        return []
    # Fix malformed files containing literal "\n" between coordinates.
    txt = txt.replace("\r", "").replace("\n", chr(10))

    polygons = []
    for raw_line in txt.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 7:
            continue

        coords = []
        for token in parts[1:]:
            try:
                coords.append(float(token))
            except Exception:
                for sub in token.replace(",", " ").replace(";", " ").split():
                    try:
                        coords.append(float(sub))
                    except Exception:
                        pass

        if len(coords) < 6:
            continue
        if len(coords) % 2 == 1:
            coords = coords[:-1]

        pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
        pts = np.clip(pts, 0.0, 1.0)
        if pts.shape[0] >= 3:
            polygons.append(pts)

    return polygons


def polygons_norm_to_mask(polygons_norm, h: int, w: int):
    mask = np.zeros((h, w), dtype=np.uint8)
    if h <= 0 or w <= 0:
        return mask

    for pts in polygons_norm:
        arr = np.asarray(pts, dtype=np.float32)
        if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
            continue

        arr_px = np.empty_like(arr)
        arr_px[:, 0] = np.clip(arr[:, 0] * (w - 1), 0, w - 1)
        arr_px[:, 1] = np.clip(arr[:, 1] * (h - 1), 0, h - 1)
        cv2.fillPoly(mask, [np.round(arr_px).astype(np.int32)], 1)

    return mask


def result_to_pred_mask(result, h: int, w: int):
    mask = np.zeros((h, w), dtype=np.uint8)
    pred_count = 0

    masks_obj = getattr(result, "masks", None)
    if masks_obj is None:
        return mask, pred_count

    polys = getattr(masks_obj, "xy", None)
    if polys is not None and len(polys) > 0:
        pred_count = int(len(polys))
        for poly in polys:
            arr = np.asarray(poly, dtype=np.float32)
            if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
                continue
            arr[:, 0] = np.clip(arr[:, 0], 0, w - 1)
            arr[:, 1] = np.clip(arr[:, 1], 0, h - 1)
            cv2.fillPoly(mask, [np.round(arr).astype(np.int32)], 1)
        return mask, pred_count

    data = getattr(masks_obj, "data", None)
    if data is None:
        return mask, pred_count

    arr = data.detach().cpu().numpy()
    pred_count = int(arr.shape[0])
    if arr.size == 0:
        return mask, pred_count

    union = (arr > 0.5).any(axis=0).astype(np.uint8)
    if union.shape != (h, w):
        union = cv2.resize(union, (w, h), interpolation=cv2.INTER_NEAREST)
    return union.astype(np.uint8), pred_count


def dice_score(pred_mask, gt_mask, eps: float = 1e-7):
    a = pred_mask.astype(bool)
    b = gt_mask.astype(bool)

    sa = float(a.sum(dtype=np.float64))
    sb = float(b.sum(dtype=np.float64))
    if sa == 0.0 and sb == 0.0:
        return 1.0

    inter = float(np.logical_and(a, b).sum(dtype=np.float64))
    return float((2.0 * inter + eps) / (sa + sb + eps))


def find_experiment_any_state(client: MlflowClient, experiment_name: str):
    for exp in client.search_experiments(view_type=ViewType.ALL):
        if exp.name == experiment_name:
            return exp
    return None


def set_or_restore_experiment(experiment_name: str):
    client = MlflowClient()
    exp = find_experiment_any_state(client, experiment_name)

    if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
        print(f"Experiment '{experiment_name}' is deleted. Restoring...")
        client.restore_experiment(exp.experiment_id)

    try:
        return mlflow.set_experiment(experiment_name)
    except MlflowException as e:
        msg = str(e).lower()
        if "deleted experiment" in msg:
            exp = find_experiment_any_state(client, experiment_name)
            if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
                client.restore_experiment(exp.experiment_id)
                return mlflow.set_experiment(experiment_name)
        raise


def resolve_checkpoint_for_training(ckpt_name: str) -> str:
    ckpt_path = Path(ckpt_name)
    if not ckpt_path.exists():
        # Let Ultralytics download by model name.
        return ckpt_name

    try:
        # Validate checkpoint with standard torch load.
        _ = torch.load(str(ckpt_path), map_location="cpu")
        return str(ckpt_path)
    except Exception:
        broken = ckpt_path.with_name(f"{ckpt_path.name}.broken")
        suffix = 1
        while broken.exists():
            broken = ckpt_path.with_name(f"{ckpt_path.name}.broken{suffix}")
            suffix += 1
        ckpt_path.rename(broken)
        print(f"[WARN] Corrupt checkpoint moved to: {broken}")
        print(f"[WARN] Will try to auto-download fresh weights for: {ckpt_name}")
        return ckpt_name


def compute_test_custom_metrics(best_model, data_yaml: Path, imgsz: int, device):
    test_img_dir, test_lbl_dir = resolve_test_split_dirs(data_yaml)
    if not test_img_dir.exists():
        raise FileNotFoundError(f"Test image dir not found: {test_img_dir}")

    test_images = [
        p for p in sorted(test_img_dir.iterdir())
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ]
    if not test_images:
        empty_metrics = {
            "dice_M_mean": float("nan"),
            "dice_M_median": float("nan"),
            "mae_count": float("nan"),
            "rmse_count": float("nan"),
            "mape_count_nonzero": float("nan"),
            "test_images_eval": 0,
            "test_images_missing_labels": 0,
        }
        return empty_metrics, pd.DataFrame()

    pred_iter = best_model.predict(
        source=str(test_img_dir),
        imgsz=imgsz,
        conf=PRED_CONF,
        iou=PRED_IOU,
        max_det=PRED_MAX_DET,
        device=device,
        stream=True,
        verbose=False,
        save=False,
    )

    rows = []
    for res in pred_iter:
        image_path = Path(res.path)
        h, w = map(int, res.orig_shape)

        label_path = test_lbl_dir / f"{image_path.stem}.txt"
        gt_polys = read_yolo_seg_polygons(label_path)
        gt_mask = polygons_norm_to_mask(gt_polys, h, w)

        pred_mask, pred_count = result_to_pred_mask(res, h, w)
        gt_count = int(len(gt_polys))
        count_error = int(pred_count - gt_count)
        abs_error = abs(count_error)
        ape = (abs_error / gt_count) if gt_count > 0 else float("nan")

        rows.append(
            {
                "image": image_path.name,
                "label_exists": int(label_path.exists()),
                "gt_count": gt_count,
                "pred_count": int(pred_count),
                "count_error": count_error,
                "count_abs_error": abs_error,
                "count_ape": float(ape),
                "dice": dice_score(pred_mask, gt_mask),
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        metrics = {
            "dice_M_mean": float("nan"),
            "dice_M_median": float("nan"),
            "mae_count": float("nan"),
            "rmse_count": float("nan"),
            "mape_count_nonzero": float("nan"),
            "test_images_eval": 0,
            "test_images_missing_labels": 0,
        }
        return metrics, df

    sq = np.square(df["count_error"].to_numpy(dtype=np.float64))
    ape_valid = df["count_ape"].dropna()

    metrics = {
        "dice_M_mean": float(df["dice"].mean()),
        "dice_M_median": float(df["dice"].median()),
        "mae_count": float(df["count_abs_error"].mean()),
        "rmse_count": float(np.sqrt(sq.mean())),
        "mape_count_nonzero": float(ape_valid.mean() * 100.0) if len(ape_valid) else float("nan"),
        "test_images_eval": int(len(df)),
        "test_images_missing_labels": int((df["label_exists"] == 0).sum()),
    }
    return metrics, df


def find_train_results_csv(train_dir: Path, project_dir: str, run_name: str):
    candidates = []
    if train_dir is not None:
        candidates.append(train_dir / "results.csv")

    proj = Path(project_dir)
    candidates.append(proj / run_name / "results.csv")
    candidates.append(Path("runs") / "segment" / proj / run_name / "results.csv")

    for base in [proj, Path("runs") / "segment" / proj, Path("runs") / "segment"]:
        if base.exists():
            try:
                candidates.extend(
                    sorted(
                        base.glob(f"**/{run_name}*/results.csv"),
                        key=lambda p: p.stat().st_mtime,
                        reverse=True,
                    )
                )
            except Exception:
                pass

    for c in candidates:
        if c is not None and c.exists():
            return c
    return None


def pick_active_results_csv(results_csv_candidates):
    existing = [p for p in results_csv_candidates if p is not None and p.exists()]
    if not existing:
        return None
    try:
        return max(existing, key=lambda p: p.stat().st_mtime)
    except Exception:
        return existing[0]


def log_partial_train_metrics_to_run(run_id: str, results_csv: Path):
    if not run_id or results_csv is None or not results_csv.exists():
        return False

    try:
        df = pd.read_csv(results_csv)
    except Exception as e:
        print(f"[WARN] Cannot read partial results.csv: {e}")
        return False

    if df.empty:
        return False

    client = MlflowClient()
    logged_any = False
    for step_idx, row in df.iterrows():
        step_raw = row.get("epoch", step_idx)
        step_float = to_float(step_raw)
        step = int(step_float) if np.isfinite(step_float) else int(step_idx)

        for k, v in row.items():
            vv = to_float(v)
            if np.isfinite(vv):
                client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
                logged_any = True

    return logged_any


def stream_train_metrics_live(run_id: str, results_csv_candidates, stop_event, poll_seconds: float = LIVE_LOG_POLL_SEC):
    """Stream new rows from Ultralytics results.csv to MLflow as train_* metrics."""
    client = MlflowClient()
    next_row = 0
    active_csv = None

    while not stop_event.is_set():
        current_csv = pick_active_results_csv(results_csv_candidates)

        if current_csv is not None and current_csv != active_csv:
            active_csv = current_csv
            next_row = 0
            print(f"[LIVE] train metrics source: {active_csv}")

        if active_csv is not None and active_csv.exists():
            try:
                df = pd.read_csv(active_csv)
            except Exception:
                if stop_event.wait(poll_seconds):
                    break
                continue

            if len(df) > next_row:
                for row_idx in range(next_row, len(df)):
                    row = df.iloc[row_idx]
                    step_raw = row.get("epoch", row_idx)
                    step_float = to_float(step_raw)
                    step = int(step_float) if np.isfinite(step_float) else int(row_idx)

                    for k, v in row.items():
                        vv = to_float(v)
                        if np.isfinite(vv):
                            client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
                next_row = len(df)

        if stop_event.wait(poll_seconds):
            break

    # Final flush in case the last rows were written right before stop.
    if active_csv is not None and active_csv.exists():
        try:
            df = pd.read_csv(active_csv)
            if len(df) > next_row:
                for row_idx in range(next_row, len(df)):
                    row = df.iloc[row_idx]
                    step_raw = row.get("epoch", row_idx)
                    step_float = to_float(step_raw)
                    step = int(step_float) if np.isfinite(step_float) else int(row_idx)

                    for k, v in row.items():
                        vv = to_float(v)
                        if np.isfinite(vv):
                            client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
        except Exception:
            pass


# Make this cell self-contained: resolve DATA_YAML if previous setup cell was not executed.
if "DATA_YAML" not in globals() or DATA_YAML is None:
    DATA_YAML = Path("cropped_736_aug_leaky/dataset/data.yaml")
else:
    DATA_YAML = Path(DATA_YAML)

if not DATA_YAML.exists():
    fallback_candidates = [
        Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
        Path("cropped_736/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
    ]
    for candidate in fallback_candidates:
        if candidate.exists():
            DATA_YAML = candidate
            break

if not DATA_YAML.exists():
    found = sorted(Path(".").rglob("cropped_736_aug_leaky/dataset/data.yaml"))
    if not found:
        found = sorted(Path(".").rglob("cropped_736/dataset/data.yaml"))
    if found:
        DATA_YAML = found[0]

if not DATA_YAML.exists():
    raise FileNotFoundError("data.yaml not found. Run dataset preparation first or set DATA_YAML manually.")

print("Using DATA_YAML:", DATA_YAML.resolve())

# Disable Ultralytics built-in MLflow callback.
# We do explicit MLflow logging in this notebook and avoid duplicate/conflicting runs.
yolo_settings.update({"mlflow": False})
print("Ultralytics setting mlflow:", yolo_settings.get("mlflow"))

# Configure tracking
if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
else:
    LOCAL_MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(LOCAL_MLFLOW_DIR.resolve().as_uri())

active_experiment = set_or_restore_experiment(MLFLOW_EXPERIMENT)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", getattr(active_experiment, "name", MLFLOW_EXPERIMENT))
print("Experiment ID:", getattr(active_experiment, "experiment_id", "unknown"))

summary_rows = []

for ckpt in MODELS_TO_TRAIN:
    model_tag = Path(ckpt).stem
    run_name = f"{model_tag}_{RUN_SUFFIX}"
    print(f"\n=== Training {ckpt} ===")

    run_id = None
    train_dir = None
    model = None
    best_model = None
    train_batch = int(TRAIN_BATCH_BY_MODEL.get(ckpt, TRAIN_BATCH))

    clear_cuda_memory(f"before {ckpt}")

    try:
        with mlflow.start_run(run_name=run_name) as active_run:
            run_id = active_run.info.run_id
            train_ckpt = resolve_checkpoint_for_training(ckpt)

            mlflow.set_tags(
                {
                    "framework": "ultralytics",
                    "task": "segment",
                    "dataset": str(DATA_YAML),
                    "model_ckpt": ckpt,
                    "offline_aug": "true",
                    "online_aug": "false",
                }
            )
            mlflow.log_metric("run_started", 1.0, step=0)
            mlflow.log_params(
                {
                    "model": ckpt,
                    "model_effective": train_ckpt,
                    "data_yaml": str(DATA_YAML),
                    "imgsz": TRAIN_IMGSZ,
                    "epochs": TRAIN_EPOCHS,
                    "batch": train_batch,
                    "workers": TRAIN_WORKERS,
                    "overlap_mask": str(TRAIN_OVERLAP_MASK).lower(),
                    "optimizer": TRAIN_OPTIMIZER,
                    "amp": str(TRAIN_AMP).lower(),
                    "patience": TRAIN_PATIENCE,
                    "project": MLFLOW_PROJECT,
                    "run_name": run_name,
                    "tracking_uri": mlflow.get_tracking_uri(),
                    "predict_conf": PRED_CONF,
                    "predict_iou": PRED_IOU,
                    "predict_max_det": PRED_MAX_DET,
                }
            )

            clear_cuda_memory(f"pre-train {ckpt}")

            expected_train_dir = Path(MLFLOW_PROJECT) / run_name
            live_results_candidates = [
                expected_train_dir / "results.csv",
                Path("runs") / "segment" / expected_train_dir / "results.csv",
                Path("runs") / "segment" / run_name / "results.csv",
            ]
            live_stop_event = threading.Event()
            live_thread = threading.Thread(
                target=stream_train_metrics_live,
                args=(run_id, live_results_candidates, live_stop_event),
                kwargs={"poll_seconds": LIVE_LOG_POLL_SEC},
                daemon=True,
            )
            live_thread.start()

            model = YOLO(train_ckpt)
            try:
                train_results = model.train(
                    data=str(DATA_YAML),
                    task="segment",
                    overlap_mask=TRAIN_OVERLAP_MASK,
                    imgsz=TRAIN_IMGSZ,
                    mask_ratio=2,
                    epochs=TRAIN_EPOCHS,
                    batch=train_batch,
                    patience=TRAIN_PATIENCE,
                    device=TRAIN_DEVICE,
                    workers=TRAIN_WORKERS,
                    optimizer=TRAIN_OPTIMIZER,
                    amp=TRAIN_AMP,
                    cache=TRAIN_CACHE,
                    project=MLFLOW_PROJECT,
                    name=run_name,
                    exist_ok=True,
                    plots=True,
                    # No online augmentation (offline-augmented dataset only)
                    degrees=0.0,
                    translate=0.0,
                    scale=0.0,
                    shear=0.0,
                    perspective=0.0,
                    fliplr=0.0,
                    flipud=0.0,
                    hsv_h=0.0,
                    hsv_s=0.0,
                    hsv_v=0.0,
                    mosaic=0.0,
                    mixup=0.0,
                    copy_paste=0.0,
                    erasing=0.0,
                )
            finally:
                live_stop_event.set()
                live_thread.join(timeout=15)

            train_dir = Path(train_results.save_dir)
            best_w = train_dir / "weights" / "best.pt"
            last_w = train_dir / "weights" / "last.pt"

            # Log train artifacts
            maybe_log_artifact(train_dir / "args.yaml", "train")
            maybe_log_artifact(train_dir / "results.csv", "train")
            maybe_log_artifact(train_dir / "results.png", "train")
            maybe_log_artifact(train_dir / "confusion_matrix.png", "train")
            maybe_log_artifact(train_dir / "confusion_matrix_normalized.png", "train")
            for nm in [
                "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png", "BoxF1_curve.png",
                "MaskPR_curve.png", "MaskP_curve.png", "MaskR_curve.png", "MaskF1_curve.png",
            ]:
                maybe_log_artifact(train_dir / nm, "train")
            maybe_log_artifact(best_w, "weights")
            maybe_log_artifact(last_w, "weights")

            # Log last epoch train metrics from results.csv if available
            r_csv = train_dir / "results.csv"
            if r_csv.exists():
                train_df = pd.read_csv(r_csv)
                if len(train_df) > 0:
                    last_row = train_df.iloc[-1].to_dict()
                    train_metrics = {}
                    for k, v in last_row.items():
                        vv = to_float(v)
                        if np.isfinite(vv):
                            train_metrics[f"train_{safe_metric_name(k)}"] = vv
                    if train_metrics:
                        mlflow.log_metrics(train_metrics)

            if not best_w.exists():
                raise FileNotFoundError(f"best.pt not found for {ckpt}: {best_w}")

            # Evaluate on test split
            best_model = YOLO(str(best_w))
            test_results = best_model.val(
                data=str(DATA_YAML),
                split="test",
                imgsz=TRAIN_IMGSZ,
                batch=1,
                workers=TRAIN_WORKERS,
                overlap_mask=TRAIN_OVERLAP_MASK,
                project=MLFLOW_PROJECT,
                name=f"{run_name}_test",
                exist_ok=True,
                plots=True,
                save_json=True,
            )

            test_dir = Path(test_results.save_dir)

            metrics_summary = {}
            metrics_summary.update(summarize_section(getattr(test_results, "box", None), "B"))
            metrics_summary.update(summarize_section(getattr(test_results, "seg", None), "M"))
            metrics_summary["fitness"] = to_float(getattr(test_results, "fitness", float("nan")))

            # Custom post-hoc metrics on test split
            custom_metrics, per_image_df = compute_test_custom_metrics(
                best_model=best_model,
                data_yaml=Path(DATA_YAML),
                imgsz=TRAIN_IMGSZ,
                device=TRAIN_DEVICE,
            )
            metrics_summary.update(custom_metrics)

            per_image_csv = test_dir / "test_per_image_metrics.csv"
            per_image_df.to_csv(per_image_csv, index=False)

            # Persist + log test metrics json
            test_json = test_dir / "test_metrics_summary.json"
            with test_json.open("w", encoding="utf-8") as f:
                json.dump(metrics_summary, f, indent=2)

            mlflow_metrics = {k: v for k, v in metrics_summary.items() if np.isfinite(v)}
            if mlflow_metrics:
                mlflow.log_metrics(mlflow_metrics)

            # Log test artifacts
            maybe_log_artifact(test_json, "test")
            maybe_log_artifact(per_image_csv, "test")
            maybe_log_artifact(test_dir / "predictions.json", "test")
            maybe_log_artifact(test_dir / "confusion_matrix.png", "test")
            maybe_log_artifact(test_dir / "confusion_matrix_normalized.png", "test")
            for nm in [
                "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png", "BoxF1_curve.png",
                "MaskPR_curve.png", "MaskP_curve.png", "MaskR_curve.png", "MaskF1_curve.png",
                "PR_curve.png", "P_curve.png", "R_curve.png", "F1_curve.png",
            ]:
                maybe_log_artifact(test_dir / nm, "test")

            mlflow.log_params(
                {
                    "ultralytics_train_dir": str(train_dir.resolve()),
                    "ultralytics_test_dir": str(test_dir.resolve()),
                    "best_weights": str(best_w.resolve()),
                }
            )

            row = {
                "model": ckpt,
                "train_dir": str(train_dir),
                "test_dir": str(test_dir),
                **metrics_summary,
            }
            summary_rows.append(row)

            print(f"Done: {ckpt}")
            print("  train_dir:", train_dir)
            print("  test_dir :", test_dir)


    except Exception as exc:
        print(f"[ERROR] {ckpt}: {exc}")

        try:
            partial_csv = find_train_results_csv(train_dir, MLFLOW_PROJECT, run_name)
            if run_id:
                client = MlflowClient()
                client.set_tag(run_id, "error_message", str(exc)[:1000])
                if partial_csv is not None:
                    logged = log_partial_train_metrics_to_run(run_id, partial_csv)
                    if logged:
                        print(f"[INFO] Logged partial train metrics from: {partial_csv}")
                    else:
                        print(f"[INFO] Partial results found but no numeric metrics: {partial_csv}")
        except Exception as log_exc:
            print(f"[WARN] Failed to log partial metrics: {log_exc}")

        summary_rows.append({"model": ckpt, "error": str(exc)})
        if STOP_ON_ERROR:
            raise
    finally:
        try:
            if best_model is not None:
                del best_model
            if model is not None:
                del model
        except Exception:
            pass
        clear_cuda_memory(f"after {ckpt}")

summary_df = pd.DataFrame(summary_rows)
try:
    display(summary_df)
except Exception:
    print(summary_df)

summary_csv = Path("mlflow_multi_model_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print("Saved summary:", summary_csv.resolve())


In [ ]:
import gc
import torch

gc.collect()

if not torch.cuda.is_available():
    print("CUDA not available")
else:
    try:
        if torch.cuda.is_initialized():
            try:
                torch.cuda.synchronize()
            except Exception:
                pass

            try:
                torch.cuda.empty_cache()
            except RuntimeError as e:
                print(f"[WARN] empty_cache failed: {e}")

            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass

        print("CUDA cache clear attempted")
    except Exception as e:
        print(f"[WARN] CUDA cleanup failed: {e}")
        print("[INFO] If this follows OOM, do Kernel Restart to reset CUDA context.")



In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
# N-only: train yolo26n-seg.pt with MLflow (imgsz=736, mask_ratio=2)
from pathlib import Path
from datetime import datetime
import os
import sys
import json
import gc

import numpy as np
import pandas as pd
import torch
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
from mlflow.exceptions import MlflowException
from ultralytics import YOLO, settings as yolo_settings

os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:64")
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "colony_yolo_seg_736"
MLFLOW_PROJECT = "runs/colony_seg_mlflow_736"

CKPT_N = "yolo26n-seg.pt"
RUN_NAME = f"yolo26n-seg_cropped736_offline_aug_mr2_nonly_{datetime.now():%Y%m%d_%H%M%S}"

TRAIN_EPOCHS = 50
TRAIN_BATCH = 1
TRAIN_IMGSZ = 736
TRAIN_PATIENCE = 80
TRAIN_DEVICE = 0
TRAIN_WORKERS = 0
TRAIN_OVERLAP_MASK = False
TRAIN_OPTIMIZER = "AdamW"
TRAIN_AMP = True
TRAIN_CACHE = False


def to_float(v):
    try:
        return float(v)
    except Exception:
        return float("nan")


def safe_metric_name(name: str) -> str:
    s = str(name).strip().replace(" ", "_")
    for bad in ["(", ")", "/", "\\", ":", ",", "|", "-", "."]:
        s = s.replace(bad, "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")


def summarize_section(section, prefix):
    if section is None:
        return {
            f"precision_{prefix}": float("nan"),
            f"recall_{prefix}": float("nan"),
            f"mAP50_{prefix}": float("nan"),
            f"mAP50_95_{prefix}": float("nan"),
        }
    return {
        f"precision_{prefix}": to_float(getattr(section, "mp", float("nan"))),
        f"recall_{prefix}": to_float(getattr(section, "mr", float("nan"))),
        f"mAP50_{prefix}": to_float(getattr(section, "map50", float("nan"))),
        f"mAP50_95_{prefix}": to_float(getattr(section, "map", float("nan"))),
    }


def find_experiment_any_state(client: MlflowClient, experiment_name: str):
    for exp in client.search_experiments(view_type=ViewType.ALL):
        if exp.name == experiment_name:
            return exp
    return None


def set_or_restore_experiment(experiment_name: str):
    client = MlflowClient()
    exp = find_experiment_any_state(client, experiment_name)
    if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
        client.restore_experiment(exp.experiment_id)
    try:
        return mlflow.set_experiment(experiment_name)
    except MlflowException as e:
        if "deleted experiment" in str(e).lower():
            exp = find_experiment_any_state(client, experiment_name)
            if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
                client.restore_experiment(exp.experiment_id)
                return mlflow.set_experiment(experiment_name)
        raise


# Resolve DATA_YAML
if "DATA_YAML" in globals() and DATA_YAML is not None:
    data_yaml = Path(DATA_YAML)
else:
    data_yaml = Path("cropped_736_aug_leaky/dataset/data.yaml")

if not data_yaml.exists():
    candidates = [
        Path("cropped_736_aug_leaky/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
        Path("cropped_736/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
    ]
    for c in candidates:
        if c.exists():
            data_yaml = c
            break

if not data_yaml.exists():
    found = sorted(Path(".").rglob("cropped_736_aug_leaky/dataset/data.yaml"))
    if found:
        data_yaml = found[0]

if not data_yaml.exists():
    raise FileNotFoundError(f"DATA_YAML not found: {data_yaml}")

# Disable Ultralytics built-in MLflow callback (we log manually)
yolo_settings.update({"mlflow": False})

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
set_or_restore_experiment(MLFLOW_EXPERIMENT)

print("DATA_YAML:", data_yaml.resolve())
print("Run name :", RUN_NAME)

# Best-effort CUDA cleanup before training
gc.collect()
if torch.cuda.is_available():
    try:
        if torch.cuda.is_initialized():
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
    except Exception:
        pass

with mlflow.start_run(run_name=RUN_NAME) as run:
    run_id = run.info.run_id
    model = None
    best_model = None
    try:
        mlflow.log_metric("run_started", 1.0, step=0)
        mlflow.log_params({
            "model": CKPT_N,
            "data_yaml": str(data_yaml),
            "imgsz": TRAIN_IMGSZ,
            "mask_ratio": 2,
            "epochs": TRAIN_EPOCHS,
            "batch": TRAIN_BATCH,
            "patience": TRAIN_PATIENCE,
            "workers": TRAIN_WORKERS,
            "device": TRAIN_DEVICE,
            "optimizer": TRAIN_OPTIMIZER,
            "amp": str(TRAIN_AMP).lower(),
            "cache": str(TRAIN_CACHE).lower(),
            "overlap_mask": str(TRAIN_OVERLAP_MASK).lower(),
            "project": MLFLOW_PROJECT,
            "run_name": RUN_NAME,
            "tracking_uri": mlflow.get_tracking_uri(),
            "online_aug": "false",
        })
        mlflow.set_tags({
            "framework": "ultralytics",
            "task": "segment",
            "dataset": str(data_yaml),
            "model_ckpt": CKPT_N,
            "offline_aug": "true",
            "online_aug": "false",
        })

        model = YOLO(CKPT_N)
        train_results = model.train(
            data=str(data_yaml),
            task="segment",
            overlap_mask=TRAIN_OVERLAP_MASK,
            imgsz=TRAIN_IMGSZ,
            mask_ratio=2,
            epochs=TRAIN_EPOCHS,
            batch=TRAIN_BATCH,
            patience=TRAIN_PATIENCE,
            device=TRAIN_DEVICE,
            workers=TRAIN_WORKERS,
            optimizer=TRAIN_OPTIMIZER,
            amp=TRAIN_AMP,
            cache=TRAIN_CACHE,
            project=MLFLOW_PROJECT,
            name=RUN_NAME,
            exist_ok=False,
            plots=True,
            # No online augmentation
            degrees=0.0,
            translate=0.0,
            scale=0.0,
            shear=0.0,
            perspective=0.0,
            fliplr=0.0,
            flipud=0.0,
            hsv_h=0.0,
            hsv_s=0.0,
            hsv_v=0.0,
            mosaic=0.0,
            mixup=0.0,
            copy_paste=0.0,
            erasing=0.0,
        )

        train_dir = Path(train_results.save_dir)
        results_csv = train_dir / "results.csv"
        best_w = train_dir / "weights" / "best.pt"
        last_w = train_dir / "weights" / "last.pt"

        # Log per-epoch metrics from results.csv
        if results_csv.exists():
            df = pd.read_csv(results_csv)
            for i, row in df.iterrows():
                step_raw = row.get("epoch", i)
                step = int(to_float(step_raw)) if np.isfinite(to_float(step_raw)) else int(i)
                for k, v in row.items():
                    vv = to_float(v)
                    if np.isfinite(vv):
                        mlflow.log_metric(f"train_{safe_metric_name(k)}", vv, step=step)
            mlflow.log_artifact(str(results_csv), artifact_path="train")

        # Train artifacts
        for p in [
            train_dir / "args.yaml",
            train_dir / "results.png",
            train_dir / "confusion_matrix.png",
            train_dir / "confusion_matrix_normalized.png",
            best_w,
            last_w,
        ]:
            if p.exists():
                mlflow.log_artifact(str(p), artifact_path="train" if p.suffix != ".pt" else "weights")

        if not best_w.exists():
            raise FileNotFoundError(f"best.pt not found: {best_w}")

        # Test split validation
        best_model = YOLO(str(best_w))
        test_results = best_model.val(
            data=str(data_yaml),
            split="test",
            imgsz=TRAIN_IMGSZ,
            batch=1,
            workers=TRAIN_WORKERS,
            overlap_mask=TRAIN_OVERLAP_MASK,
            project=MLFLOW_PROJECT,
            name=f"{RUN_NAME}_test",
            exist_ok=True,
            plots=True,
            save_json=True,
        )
        test_dir = Path(test_results.save_dir)

        metrics_summary = {}
        metrics_summary.update(summarize_section(getattr(test_results, "box", None), "B"))
        metrics_summary.update(summarize_section(getattr(test_results, "seg", None), "M"))
        metrics_summary["fitness"] = to_float(getattr(test_results, "fitness", float("nan")))

        mlflow.log_metrics({k: v for k, v in metrics_summary.items() if np.isfinite(v)})

        test_json = test_dir / "test_metrics_summary.json"
        with test_json.open("w", encoding="utf-8") as f:
            json.dump(metrics_summary, f, indent=2)

        for p in [
            test_json,
            test_dir / "predictions.json",
            test_dir / "confusion_matrix.png",
            test_dir / "confusion_matrix_normalized.png",
        ]:
            if p.exists():
                mlflow.log_artifact(str(p), artifact_path="test")

        mlflow.log_params({
            "ultralytics_train_dir": str(train_dir.resolve()),
            "ultralytics_test_dir": str(test_dir.resolve()),
            "best_weights": str(best_w.resolve()),
        })

        print("DONE:", RUN_NAME)
        print("train_dir:", train_dir)
        print("test_dir :", test_dir)

    except Exception as e:
        MlflowClient().set_tag(run_id, "error_message", str(e)[:1000])
        print("[ERROR]", e)
        raise
    finally:
        try:
            if best_model is not None:
                del best_model
            if model is not None:
                del model
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            try:
                if torch.cuda.is_initialized():
                    torch.cuda.empty_cache()
                    torch.cuda.ipc_collect()
            except Exception:
                pass


In [ ]:
# Plot curves from partial training (works even if training was interrupted)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def resolve_results_csv_for_plot():
    candidates = []

    # 1) Prefer current RUN_NAME if available
    run_name = globals().get("RUN_NAME")
    project_hint = globals().get("MLFLOW_PROJECT", "runs/colony_seg_mlflow_736")

    if run_name:
        direct = [
            Path(project_hint) / run_name / "results.csv",
            Path("runs") / "segment" / Path(project_hint) / run_name / "results.csv",
            Path("runs") / "segment" / "runs" / Path(project_hint).name / run_name / "results.csv",
            Path("runs") / "segment" / "runs" / "colony_seg_mlflow_736" / run_name / "results.csv",
        ]
        for p in direct:
            if p.exists():
                candidates.append(p)

    # 2) Search latest in known project dirs
    bases = [
        Path("runs") / "segment" / "runs" / "colony_seg_mlflow_736",
        Path("runs") / "segment" / Path(project_hint),
        Path(project_hint),
    ]
    for base in bases:
        if base.exists():
            for d in base.iterdir():
                if not d.is_dir() or d.name.endswith("_test"):
                    continue
                p = d / "results.csv"
                if p.exists():
                    candidates.append(p)

    if not candidates:
        raise FileNotFoundError("No results.csv found for plotting.")

    candidates = sorted(set(candidates), key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]


results_csv = resolve_results_csv_for_plot()
run_dir = results_csv.parent
print("Using results:", results_csv.resolve())

df = pd.read_csv(results_csv)
if df.empty:
    raise RuntimeError("results.csv is empty")

# Robust epoch axis
if "epoch" in df.columns:
    x = pd.to_numeric(df["epoch"], errors="coerce")
    if x.isna().all():
        x = pd.Series(np.arange(1, len(df) + 1), dtype=float)
else:
    x = pd.Series(np.arange(1, len(df) + 1), dtype=float)

# If epoch starts at 0, show 1-based on chart labels
if float(x.min()) == 0.0:
    x_plot = x + 1.0
else:
    x_plot = x

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1) Train losses
ax = axes[0, 0]
for col in ["train/box_loss", "train/seg_loss", "train/cls_loss", "train/dfl_loss", "train/sem_loss"]:
    if col in df.columns:
        ax.plot(x_plot, df[col], label=col.replace("train/", ""), linewidth=2)
ax.set_title("Train Losses")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)
if ax.lines:
    ax.legend(loc="best")

# 2) Val losses
ax = axes[0, 1]
for col in ["val/box_loss", "val/seg_loss", "val/cls_loss", "val/dfl_loss", "val/sem_loss"]:
    if col in df.columns:
        ax.plot(x_plot, df[col], label=col.replace("val/", ""), linewidth=2)
ax.set_title("Val Losses")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)
if ax.lines:
    ax.legend(loc="best")

# 3) Segmentation quality metrics
ax = axes[1, 0]
for col in ["metrics/mAP50(M)", "metrics/mAP50-95(M)", "metrics/precision(M)", "metrics/recall(M)"]:
    if col in df.columns:
        ax.plot(x_plot, df[col], label=col.replace("metrics/", ""), linewidth=2)
ax.set_title("Segmentation Metrics")
ax.set_xlabel("Epoch")
ax.set_ylabel("Value")
ax.grid(True, alpha=0.3)
if ax.lines:
    ax.legend(loc="best")

# 4) Learning rates
ax = axes[1, 1]
for col in ["lr/pg0", "lr/pg1", "lr/pg2"]:
    if col in df.columns:
        ax.plot(x_plot, df[col], label=col, linewidth=2)
ax.set_title("Learning Rate Schedules")
ax.set_xlabel("Epoch")
ax.set_ylabel("LR")
ax.grid(True, alpha=0.3)
if ax.lines:
    ax.legend(loc="best")

plt.tight_layout()
fig_path = run_dir / "partial_training_curves.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

# Print last available epoch summary (useful for interrupted runs)
last = df.iloc[-1]
epoch_last = int(x_plot.iloc[-1])
print(f"Last logged epoch: {epoch_last}")
for col in ["train/box_loss", "train/seg_loss", "train/cls_loss", "train/dfl_loss", "train/sem_loss", "metrics/mAP50(M)", "metrics/mAP50-95(M)"]:
    if col in df.columns:
        print(f"{col}: {float(last[col]):.6f}")

if len(df) < 50:
    print(f"[INFO] Training appears interrupted or not finished: only {len(df)} epochs logged.")

print("Curves saved to:", fig_path.resolve())



In [ ]:
# # X-only: train yolo26x-seg.pt in same MLflow experiment/project
# from pathlib import Path
# from datetime import datetime
# import os, sys
# import numpy as np
# import pandas as pd
# import torch
# import mlflow
# from mlflow.tracking import MlflowClient
# from mlflow.entities import ViewType
# from mlflow.exceptions import MlflowException
# from ultralytics import YOLO, settings as yolo_settings

# os.environ.setdefault("PYTHONIOENCODING", "utf-8")
# try:
#     sys.stdout.reconfigure(encoding="utf-8")
#     sys.stderr.reconfigure(encoding="utf-8")
# except Exception:
#     pass

# MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
# MLFLOW_EXPERIMENT = "colony_yolo_seg"
# MLFLOW_PROJECT = "runs/colony_seg_mlflow"

# CKPT_X = "yolo26x-seg.pt"  # если файла нет, Ultralytics попытается скачать
# RUN_NAME = f"yolo26x-seg_cropped720_offline_aug_mr1_xonly_{datetime.now():%Y%m%d_%H%M%S}"

# TRAIN_EPOCHS = 50
# TRAIN_BATCH = 8
# TRAIN_IMGSZ = 720
# TRAIN_PATIENCE = 80
# TRAIN_DEVICE = None

# def to_float(v):
#     try:
#         return float(v)
#     except Exception:
#         return float("nan")

# def safe_metric_name(name: str) -> str:
#     s = str(name).strip().replace(" ", "_")
#     for bad in ["(", ")", "/", "\\", ":", ",", "|", "-", "."]:
#         s = s.replace(bad, "_")
#     while "__" in s:
#         s = s.replace("__", "_")
#     return s.strip("_")

# def summarize_section(section, prefix):
#     if section is None:
#         return {
#             f"precision_{prefix}": float("nan"),
#             f"recall_{prefix}": float("nan"),
#             f"mAP50_{prefix}": float("nan"),
#             f"mAP50_95_{prefix}": float("nan"),
#         }
#     return {
#         f"precision_{prefix}": to_float(getattr(section, "mp", float("nan"))),
#         f"recall_{prefix}": to_float(getattr(section, "mr", float("nan"))),
#         f"mAP50_{prefix}": to_float(getattr(section, "map50", float("nan"))),
#         f"mAP50_95_{prefix}": to_float(getattr(section, "map", float("nan"))),
#     }

# def find_experiment_any_state(client: MlflowClient, experiment_name: str):
#     for exp in client.search_experiments(view_type=ViewType.ALL):
#         if exp.name == experiment_name:
#             return exp
#     return None

# def set_or_restore_experiment(experiment_name: str):
#     client = MlflowClient()
#     exp = find_experiment_any_state(client, experiment_name)
#     if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
#         client.restore_experiment(exp.experiment_id)
#     try:
#         return mlflow.set_experiment(experiment_name)
#     except MlflowException as e:
#         if "deleted experiment" in str(e).lower():
#             exp = find_experiment_any_state(client, experiment_name)
#             if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
#                 client.restore_experiment(exp.experiment_id)
#                 return mlflow.set_experiment(experiment_name)
#         raise

# # DATA_YAML fallback
# if "DATA_YAML" not in globals() or DATA_YAML is None:
#     DATA_YAML = Path("cropped_720_aug_leaky/dataset/data.yaml")
# else:
#     DATA_YAML = Path(DATA_YAML)

# if not DATA_YAML.exists():
#     raise FileNotFoundError(f"DATA_YAML not found: {DATA_YAML}")

# # disable built-in Ultralytics mlflow callback
# yolo_settings.update({"mlflow": False})

# mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
# set_or_restore_experiment(MLFLOW_EXPERIMENT)

# print("DATA_YAML:", DATA_YAML.resolve())
# print("Run name :", RUN_NAME)

# with mlflow.start_run(run_name=RUN_NAME) as run:
#     run_id = run.info.run_id
#     mlflow.log_metric("run_started", 1.0, step=0)
#     mlflow.log_params({
#         "model": CKPT_X,
#         "data_yaml": str(DATA_YAML),
#         "imgsz": TRAIN_IMGSZ,
#         "epochs": TRAIN_EPOCHS,
#         "batch": TRAIN_BATCH,
#         "patience": TRAIN_PATIENCE,
#         "project": MLFLOW_PROJECT,
#         "run_name": RUN_NAME,
#         "tracking_uri": mlflow.get_tracking_uri(),
#         "online_aug": "false",
#     })
#     mlflow.set_tags({
#         "framework": "ultralytics",
#         "task": "segment",
#         "dataset": str(DATA_YAML),
#         "model_ckpt": CKPT_X,
#         "offline_aug": "true",
#         "online_aug": "false",
#     })

#     try:
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()

#         model = YOLO(CKPT_X)
#         train_results = model.train(
#             data=str(DATA_YAML),
#             task="segment",
#             imgsz=TRAIN_IMGSZ,
#             mask_ratio=1,
#             epochs=TRAIN_EPOCHS,
#             batch=TRAIN_BATCH,
#             patience=TRAIN_PATIENCE,
#             device=TRAIN_DEVICE,
#             project=MLFLOW_PROJECT,
#             name=RUN_NAME,
#             exist_ok=False,
#             plots=True,
#             degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
#             fliplr=0.0, flipud=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
#             mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
#         )

#         train_dir = Path(train_results.save_dir)
#         results_csv = train_dir / "results.csv"
#         best_w = train_dir / "weights" / "best.pt"
#         last_w = train_dir / "weights" / "last.pt"

#         # online-like logging from csv (all epochs)
#         if results_csv.exists():
#             df = pd.read_csv(results_csv)
#             for i, row in df.iterrows():
#                 step_raw = row.get("epoch", i)
#                 step = int(to_float(step_raw)) if np.isfinite(to_float(step_raw)) else int(i)
#                 for k, v in row.items():
#                     vv = to_float(v)
#                     if np.isfinite(vv):
#                         mlflow.log_metric(f"train_{safe_metric_name(k)}", vv, step=step)
#             mlflow.log_artifact(str(results_csv), artifact_path="train")

#         # train artifacts
#         for p in [
#             train_dir / "args.yaml",
#             train_dir / "results.png",
#             train_dir / "confusion_matrix.png",
#             train_dir / "confusion_matrix_normalized.png",
#             best_w, last_w
#         ]:
#             if p.exists():
#                 mlflow.log_artifact(str(p), artifact_path="train" if p.suffix != ".pt" else "weights")

#         if not best_w.exists():
#             raise FileNotFoundError(f"best.pt not found: {best_w}")

#         # test validation
#         best_model = YOLO(str(best_w))
#         test_results = best_model.val(
#             data=str(DATA_YAML),
#             split="test",
#             imgsz=TRAIN_IMGSZ,
#             project=MLFLOW_PROJECT,
#             name=f"{RUN_NAME}_test",
#             exist_ok=True,
#             plots=True,
#             save_json=True,
#         )
#         test_dir = Path(test_results.save_dir)

#         metrics_summary = {}
#         metrics_summary.update(summarize_section(getattr(test_results, "box", None), "B"))
#         metrics_summary.update(summarize_section(getattr(test_results, "seg", None), "M"))
#         metrics_summary["fitness"] = to_float(getattr(test_results, "fitness", float("nan")))

#         mlflow.log_metrics({k: v for k, v in metrics_summary.items() if np.isfinite(v)})

#         test_json = test_dir / "test_metrics_summary.json"
#         import json
#         with test_json.open("w", encoding="utf-8") as f:
#             json.dump(metrics_summary, f, indent=2)

#         for p in [
#             test_json,
#             test_dir / "predictions.json",
#             test_dir / "confusion_matrix.png",
#             test_dir / "confusion_matrix_normalized.png",
#         ]:
#             if p.exists():
#                 mlflow.log_artifact(str(p), artifact_path="test")

#         mlflow.log_params({
#             "ultralytics_train_dir": str(train_dir.resolve()),
#             "ultralytics_test_dir": str(test_dir.resolve()),
#             "best_weights": str(best_w.resolve()),
#         })

#         print("DONE:", RUN_NAME)
#         print("train_dir:", train_dir)
#         print("test_dir :", test_dir)

#     except Exception as e:
#         MlflowClient().set_tag(run_id, "error_message", str(e)[:1000])
#         print("[ERROR]", e)
#         raise




In [ ]:
# Preview segmentation masks and contours only (no bounding boxes) for all trained YOLO models
from pathlib import Path
import re
import colorsys

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
PREVIEW_IMAGE_PATHS = [
    Path("cropped_736/images/train/1127763_IMG_4363.jpg"),
    Path("cropped_736/images/train/1127763_IMG_4615.jpg"),
    Path("cropped_736/images/train/1127763_IMG_4672.jpg"),
    Path("cropped_736/images/train/1127763_IMG_6137.jpg"),
]
PRED_CONF = 0.25
PRED_IOU = 0.60
MASK_ALPHA = 0.55
COLS = 2
CONTOUR_THICKNESS = 2
SMOOTH_SIGMA = 0.9
SAVE_PREVIEW_PNG = True
PREVIEW_SAVE_DIR = Path("runs/segment/runs/colony_seg_mlflow/preview_masks_only")


MODEL_ORDER = ["yolo26n-seg", "yolo26s-seg", "yolo26m-seg", "yolo26l-seg", "yolo26x-seg"]
MODEL_TAG_RE = re.compile(r"^(yolo26[a-z]-seg)", re.IGNORECASE)


if SAVE_PREVIEW_PNG:
    PREVIEW_SAVE_DIR.mkdir(parents=True, exist_ok=True)


def resolve_preview_images():
    selected = []
    missing = []

    for rel_path in PREVIEW_IMAGE_PATHS:
        p = Path(rel_path)
        if p.exists():
            selected.append(p)
            continue

        p_abs = Path("C:/ColonyNet") / p
        if p_abs.exists():
            selected.append(p_abs)
        else:
            missing.append(str(rel_path))

    if missing:
        raise FileNotFoundError("Missing preview images:" + "\n" + "\n".join(missing))

    return selected



def collect_trained_best_weights():
    by_model = {}

    # Include current globals if available
    extra_candidates = []
    if "best_weights" in globals():
        try:
            p = Path(best_weights)
            if p.exists():
                extra_candidates.append(p)
        except Exception:
            pass

    if "run_dir" in globals():
        try:
            p = Path(run_dir) / "weights" / "best.pt"
            if p.exists():
                extra_candidates.append(p)
        except Exception:
            pass

    for p in extra_candidates:
        run_name = p.parent.parent.name
        m = MODEL_TAG_RE.match(run_name)
        if not m:
            continue
        model_tag = m.group(1).lower()
        prev = by_model.get(model_tag)
        if prev is None or p.stat().st_mtime > prev[1].stat().st_mtime:
            by_model[model_tag] = (run_name, p)

    bases = [
        Path("runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
    ]

    for base in bases:
        if not base.exists():
            continue
        for d in base.iterdir():
            if not d.is_dir() or d.name.endswith("_test"):
                continue
            best = d / "weights" / "best.pt"
            if not best.exists():
                continue

            m = MODEL_TAG_RE.match(d.name)
            if not m:
                continue
            model_tag = m.group(1).lower()

            prev = by_model.get(model_tag)
            if prev is None or best.stat().st_mtime > prev[1].stat().st_mtime:
                by_model[model_tag] = (d.name, best)

    if not by_model:
        raise FileNotFoundError("No trained best.pt found in colony_seg_mlflow runs.")

    ordered = []
    for tag in MODEL_ORDER:
        if tag in by_model:
            run_name, best = by_model[tag]
            ordered.append((tag, run_name, best))

    # Append any extra tags not in MODEL_ORDER
    for tag, (run_name, best) in sorted(by_model.items()):
        if tag not in MODEL_ORDER:
            ordered.append((tag, run_name, best))

    return ordered


def color_for_idx(i, n_total=1):
    # High-contrast deterministic instance colors.
    if n_total < 1:
        n_total = 1
    t = ((i * 37) % n_total) / max(1, n_total - 1)
    hue = (0.02 + 0.96 * t) % 1.0
    sat = 0.95
    val = 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
    return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


def render_masks_only(image_path, result):
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        return None, 0

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    overlay = rgb.copy()
    h, w = overlay.shape[:2]

    n_masks = 0
    if result.masks is not None and getattr(result.masks, "data", None) is not None:
        mask_data = result.masks.data
        if hasattr(mask_data, "detach"):
            mask_stack = mask_data.detach().cpu().numpy()
        else:
            mask_stack = np.asarray(mask_data)

        n_total = len(mask_stack)
        for k, mask_prob in enumerate(mask_stack):
            if mask_prob is None:
                continue

            mask_prob = np.asarray(mask_prob, dtype=np.float32)
            if mask_prob.ndim != 2:
                continue

            if mask_prob.shape != (h, w):
                mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

            # Soft mask edges for smoother visualization.
            mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
            mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
            if float(mask_alpha.max()) < 0.01:
                continue

            color = color_for_idx(k, n_total=n_total)
            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            binary = (mask_prob >= 0.5).astype(np.uint8)
            if binary.any():
                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                if contours:
                    cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

    elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
        # Fallback to polygon mode if data masks are unavailable.
        polys = result.masks.xy
        n_total = len(polys)
        for k, poly in enumerate(polys):
            if poly is None or len(poly) < 3:
                continue

            pts = np.round(poly).astype(np.int32)
            color = color_for_idx(k, n_total=n_total)

            mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
            n_masks += 1

    out = np.clip(overlay, 0, 255).astype(np.uint8)
    return out, n_masks


def id_from_name(path: Path) -> str:
    m = re.search(r"(\d+)(?!.*\d)", path.stem)
    return m.group(1) if m else path.stem


weights_info = collect_trained_best_weights()
preview_images = resolve_preview_images()

print("Selected images:")
for p in preview_images:
    print(f"- {p}")

print("Models for preview:")
for tag, run_name, w in weights_info:
    print(f"- {tag}: {w} (run: {run_name})")

id_tag = "_".join(id_from_name(p) for p in preview_images)

for tag, run_name, weights_path in weights_info:
    print(f"\nPreviewing {tag} from: {weights_path}")
    model = YOLO(str(weights_path))
    results = model.predict(
        source=[str(p) for p in preview_images],
        conf=PRED_CONF,
        iou=PRED_IOU,
        save=False,
        retina_masks=True,
        verbose=False,
    )

    n = len(preview_images)
    rows = int(np.ceil(n / COLS))
    fig, axes = plt.subplots(rows, COLS, figsize=(6 * COLS, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, img_path, res in zip(axes, preview_images, results):
        out, n_masks = render_masks_only(img_path, res)
        if out is None:
            ax.set_title(f"{img_path.name} (read error)", fontsize=9)
            continue
        ax.imshow(out)
        ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
        ax.axis("off")

    fig.suptitle(f"{tag} | {run_name}", fontsize=14)
    plt.tight_layout()

    if SAVE_PREVIEW_PNG:
        out_png = PREVIEW_SAVE_DIR / f"{tag}_{run_name}_ids_{id_tag}_preview_v2.png"
        fig.savefig(out_png, dpi=180, bbox_inches="tight")
        print(f"Saved preview: {out_png}")

    plt.show()

    # Free memory between models
    del model
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"[WARN] empty_cache failed after preview: {e}")








In [ ]:
# Summary table: all metrics for YOLO models from MLflow
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "colony_yolo_seg"
MODEL_ORDER = [
    "yolo26n-seg.pt",
    "yolo26s-seg.pt",
    "yolo26m-seg.pt",
    "yolo26l-seg.pt",
    "yolo26x-seg.pt",
]
PREFER_FINISHED = True  # if True, pick latest FINISHED run per model; fallback to latest any status


mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
if exp is None:
    raise FileNotFoundError(f"MLflow experiment not found: {MLFLOW_EXPERIMENT}")

runs = client.search_runs(
    [exp.experiment_id],
    run_view_type=ViewType.ALL,
    max_results=1000,
    order_by=["attributes.start_time DESC"],
)


def extract_model_key(run):
    model_param = str(run.data.params.get("model", "")).lower()
    run_name = str(run.data.tags.get("mlflow.runName", "")).lower()

    m = re.search(r"(yolo26[nsmlx]-seg\.pt)", model_param)
    if m:
        return m.group(1)

    m = re.search(r"(yolo26[nsmlx]-seg)", run_name)
    if m:
        return m.group(1) + ".pt"

    return None


by_model_any = {}
by_model_finished = {}
for run in runs:
    key = extract_model_key(run)
    if key not in MODEL_ORDER:
        continue
    if key not in by_model_any:
        by_model_any[key] = run
    if run.info.status == "FINISHED" and key not in by_model_finished:
        by_model_finished[key] = run

selected = {}
for key in MODEL_ORDER:
    if PREFER_FINISHED and key in by_model_finished:
        selected[key] = by_model_finished[key]
    elif key in by_model_any:
        selected[key] = by_model_any[key]

if not selected:
    raise RuntimeError("No matching YOLO runs found in MLflow.")

metric_keys = sorted({k for run in selected.values() for k in run.data.metrics.keys()})
rows = []
now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)

for key in MODEL_ORDER:
    run = selected.get(key)
    if run is None:
        rows.append({"model": key, "status": "NOT_FOUND"})
        continue

    start_ms = run.info.start_time or np.nan
    end_ms = run.info.end_time if run.info.end_time is not None else now_ms

    row = {
        "model": key,
        "run_name": run.data.tags.get("mlflow.runName", ""),
        "status": run.info.status,
        "run_id": run.info.run_id,
        "start_time": pd.to_datetime(start_ms, unit="ms", utc=True) if pd.notna(start_ms) else pd.NaT,
        "duration_min": (end_ms - start_ms) / 60000.0 if pd.notna(start_ms) else np.nan,
        "error_message": run.data.tags.get("error_message", ""),
    }

    for mk in metric_keys:
        row[mk] = run.data.metrics.get(mk, np.nan)

    rows.append(row)

summary_df = pd.DataFrame(rows)

base_cols = ["model", "run_name", "status", "duration_min", "run_id", "start_time", "error_message"]
metric_cols = [c for c in summary_df.columns if c not in base_cols]
summary_df = summary_df[base_cols + metric_cols]

# Optional rounded view for readability
display_df = summary_df.copy()
for c in display_df.columns:
    if pd.api.types.is_numeric_dtype(display_df[c]):
        display_df[c] = display_df[c].round(6)

display(display_df)

out_csv = Path("mlflow_models_metrics_summary_latest.csv")
summary_df.to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_csv.resolve())



In [ ]:
# Recalculate custom metrics for yolo26x-seg.pt without retraining (Dice/MAE/RMSE/MAPE)
from pathlib import Path
import json
import numpy as np
import pandas as pd
import cv2
import yaml
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
from ultralytics import YOLO

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "colony_yolo_seg"
X_MODEL_KEY = "yolo26x-seg.pt"
LOG_RECALC_TO_MLFLOW = True  # set False if you only need local files/print

PRED_CONF = 0.25
PRED_IOU = 0.7
PRED_MAX_DET = 300
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def resolve_data_yaml_local():
    if "DATA_YAML" in globals() and DATA_YAML is not None:
        p = Path(DATA_YAML)
        if p.exists():
            return p

    for c in [
        Path("cropped_736_aug_leaky/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
        Path("cropped_736/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
    ]:
        if c.exists():
            return c

    found = sorted(Path(".").rglob("cropped_736_aug_leaky/dataset/data.yaml"))
    if not found:
        found = sorted(Path(".").rglob("cropped_736/dataset/data.yaml"))
    if found:
        return found[0]

    raise FileNotFoundError("data.yaml not found")


def resolve_latest_x_run_and_weights():
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    client = MlflowClient()
    exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
    if exp is None:
        raise FileNotFoundError(f"Experiment not found: {MLFLOW_EXPERIMENT}")

    runs = client.search_runs(
        [exp.experiment_id],
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=1000,
        order_by=["attributes.start_time DESC"],
    )

    selected = None
    for r in runs:
        model_param = str(r.data.params.get("model", "")).lower()
        run_name = str(r.data.tags.get("mlflow.runName", "")).lower()
        if X_MODEL_KEY in model_param or "yolo26x-seg" in run_name:
            selected = r
            break

    if selected is None:
        raise RuntimeError("No MLflow run found for yolo26x-seg")

    candidates = []
    train_dir_param = selected.data.params.get("ultralytics_train_dir")
    if train_dir_param:
        candidates.append(Path(train_dir_param) / "weights" / "best.pt")

    for base in [
        Path("runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
    ]:
        if not base.exists():
            continue
        for d in sorted(base.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if not d.is_dir() or d.name.endswith("_test"):
                continue
            if "yolo26x-seg" not in d.name.lower():
                continue
            candidates.append(d / "weights" / "best.pt")

    for c in candidates:
        if c.exists():
            return selected, c

    raise FileNotFoundError("best.pt for yolo26x-seg not found")


def resolve_test_split_dirs(data_yaml_path: Path):
    data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8")) or {}
    test_raw = data.get("test")
    if not test_raw:
        raise KeyError("`test` path is missing in data.yaml")

    test_img_dir = Path(str(test_raw).strip().strip('"').strip("'"))
    if not test_img_dir.is_absolute():
        test_img_dir = (data_yaml_path.parent / test_img_dir).resolve()

    split_name = test_img_dir.name
    label_candidates = [data_yaml_path.parent / "labels" / split_name]
    if test_img_dir.parent.name == "images":
        label_candidates.append(test_img_dir.parent.parent / "labels" / split_name)
    label_candidates.append(Path(str(test_img_dir).replace("\\images\\", "\\labels\\").replace("/images/", "/labels/")))

    for cand in label_candidates:
        if cand.exists():
            return test_img_dir, cand

    return test_img_dir, label_candidates[0]


def read_yolo_seg_polygons(label_path: Path):
    if not label_path.exists():
        return []

    txt = label_path.read_text(encoding="utf-8", errors="ignore")
    if not txt.strip():
        return []

    txt = txt.replace("\r", "").replace("\\n", "\n")

    polygons = []
    for raw_line in txt.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 7:
            continue

        coords = []
        for token in parts[1:]:
            try:
                coords.append(float(token))
            except Exception:
                for sub in token.replace(",", " ").replace(";", " ").split():
                    try:
                        coords.append(float(sub))
                    except Exception:
                        pass

        if len(coords) < 6:
            continue
        if len(coords) % 2 == 1:
            coords = coords[:-1]

        pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
        pts = np.clip(pts, 0.0, 1.0)
        if pts.shape[0] >= 3:
            polygons.append(pts)

    return polygons


def polygons_norm_to_mask(polygons_norm, h: int, w: int):
    mask = np.zeros((h, w), dtype=np.uint8)
    if h <= 0 or w <= 0:
        return mask

    for pts in polygons_norm:
        arr = np.asarray(pts, dtype=np.float32)
        if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
            continue

        arr_px = np.empty_like(arr)
        arr_px[:, 0] = np.clip(arr[:, 0] * (w - 1), 0, w - 1)
        arr_px[:, 1] = np.clip(arr[:, 1] * (h - 1), 0, h - 1)
        cv2.fillPoly(mask, [np.round(arr_px).astype(np.int32)], 1)

    return mask


def result_to_pred_mask(result, h: int, w: int):
    mask = np.zeros((h, w), dtype=np.uint8)
    pred_count = 0

    masks_obj = getattr(result, "masks", None)
    if masks_obj is None:
        return mask, pred_count

    polys = getattr(masks_obj, "xy", None)
    if polys is not None and len(polys) > 0:
        pred_count = int(len(polys))
        for poly in polys:
            arr = np.asarray(poly, dtype=np.float32)
            if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
                continue
            arr[:, 0] = np.clip(arr[:, 0], 0, w - 1)
            arr[:, 1] = np.clip(arr[:, 1], 0, h - 1)
            cv2.fillPoly(mask, [np.round(arr).astype(np.int32)], 1)
        return mask, pred_count

    data = getattr(masks_obj, "data", None)
    if data is None:
        return mask, pred_count

    arr = data.detach().cpu().numpy()
    pred_count = int(arr.shape[0])
    if arr.size == 0:
        return mask, pred_count

    union = (arr > 0.5).any(axis=0).astype(np.uint8)
    if union.shape != (h, w):
        union = cv2.resize(union, (w, h), interpolation=cv2.INTER_NEAREST)
    return union.astype(np.uint8), pred_count


def dice_score(pred_mask, gt_mask, eps: float = 1e-7):
    a = pred_mask.astype(bool)
    b = gt_mask.astype(bool)
    sa = float(a.sum(dtype=np.float64))
    sb = float(b.sum(dtype=np.float64))
    if sa == 0.0 and sb == 0.0:
        return 1.0
    inter = float(np.logical_and(a, b).sum(dtype=np.float64))
    return float((2.0 * inter + eps) / (sa + sb + eps))


def compute_custom_metrics(best_model, data_yaml: Path, imgsz: int = 736):
    test_img_dir, test_lbl_dir = resolve_test_split_dirs(data_yaml)
    test_images = [p for p in sorted(test_img_dir.iterdir()) if p.is_file() and p.suffix.lower() in IMG_EXTS]
    if not test_images:
        raise RuntimeError(f"No test images in: {test_img_dir}")

    pred_iter = best_model.predict(
        source=str(test_img_dir),
        imgsz=imgsz,
        conf=PRED_CONF,
        iou=PRED_IOU,
        max_det=PRED_MAX_DET,
        stream=True,
        verbose=False,
        save=False,
    )

    rows = []
    for res in pred_iter:
        image_path = Path(res.path)
        h, w = map(int, res.orig_shape)

        label_path = test_lbl_dir / f"{image_path.stem}.txt"
        gt_polys = read_yolo_seg_polygons(label_path)
        gt_mask = polygons_norm_to_mask(gt_polys, h, w)

        pred_mask, pred_count = result_to_pred_mask(res, h, w)
        gt_count = int(len(gt_polys))
        count_error = int(pred_count - gt_count)
        abs_error = abs(count_error)
        ape = (abs_error / gt_count) if gt_count > 0 else np.nan

        rows.append({
            "image": image_path.name,
            "label_exists": int(label_path.exists()),
            "gt_count": gt_count,
            "pred_count": int(pred_count),
            "count_error": count_error,
            "count_abs_error": abs_error,
            "count_ape": float(ape),
            "dice": dice_score(pred_mask, gt_mask),
        })

    df = pd.DataFrame(rows)
    sq = np.square(df["count_error"].to_numpy(dtype=np.float64))
    ape_valid = df["count_ape"].dropna()

    metrics = {
        "dice_M_mean": float(df["dice"].mean()),
        "dice_M_median": float(df["dice"].median()),
        "mae_count": float(df["count_abs_error"].mean()),
        "rmse_count": float(np.sqrt(sq.mean())),
        "mape_count_nonzero": float(ape_valid.mean() * 100.0) if len(ape_valid) else np.nan,
        "test_images_eval": int(len(df)),
        "test_images_missing_labels": int((df["label_exists"] == 0).sum()),
    }
    return metrics, df, test_img_dir


# ---- run ----
data_yaml = resolve_data_yaml_local()
run_x, best_weights_x = resolve_latest_x_run_and_weights()
print("Using DATA_YAML:", data_yaml.resolve())
print("Using X run:", run_x.data.tags.get("mlflow.runName"), run_x.info.run_id)
print("Using weights:", best_weights_x)

best_model_x = YOLO(str(best_weights_x))
custom_metrics, per_image_df, used_test_dir = compute_custom_metrics(best_model_x, data_yaml, imgsz=736)

print("\nCustom metrics (X, recalculated):")
for k, v in custom_metrics.items():
    print(f"- {k}: {v}")

try:
    display(per_image_df.head(10))
except Exception:
    print(per_image_df.head(10))

# Save local artifacts
out_dir = Path("runs/segment/runs/colony_seg_mlflow/x_metrics_recalc")
out_dir.mkdir(parents=True, exist_ok=True)
out_json = out_dir / "x_custom_metrics_recalc.json"
out_csv = out_dir / "x_per_image_metrics_recalc.csv"
out_json.write_text(json.dumps(custom_metrics, indent=2), encoding="utf-8")
per_image_df.to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_json.resolve())
print("Saved:", out_csv.resolve())

# Optionally log to MLflow as separate run (no retraining)
if LOG_RECALC_TO_MLFLOW:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    recalc_name = f"{run_x.data.tags.get('mlflow.runName','yolo26x')}_custom_metrics_recalc"
    with mlflow.start_run(run_name=recalc_name):
        mlflow.set_tags({
            "task": "segment_custom_metrics_recalc",
            "source_run_id": run_x.info.run_id,
            "source_model": X_MODEL_KEY,
            "retrain": "false",
        })
        mlflow.log_params({
            "data_yaml": str(data_yaml),
            "weights": str(best_weights_x),
            "test_dir": str(used_test_dir),
            "pred_conf": PRED_CONF,
            "pred_iou": PRED_IOU,
            "pred_max_det": PRED_MAX_DET,
        })
        mlflow.log_metrics({k: float(v) for k, v in custom_metrics.items() if pd.notna(v)})
        mlflow.log_artifact(str(out_json), artifact_path="recalc")
        mlflow.log_artifact(str(out_csv), artifact_path="recalc")

    print("Logged recalculated metrics to MLflow as separate run.")




In [ ]:
# Predict on full cropped_736/images/train using latest yolo26n-seg best weights
from pathlib import Path
from datetime import datetime
import colorsys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SOURCE_DIR = Path("cropped_736/images/train")
TARGET_PREVIEW_FILES = [
    "1127763_IMG_4363.jpg",
    "1127763_IMG_4615.jpg",
    "1127763_IMG_4672.jpg",
    "1127763_IMG_6137.jpg",
]
PRED_CONF = 0.25
PRED_IOU = 0.6
PRED_IMGSZ = 736
MASK_ALPHA = 0.55
CONTOUR_THICKNESS = 2
SMOOTH_SIGMA = 0.9


def resolve_n_best_weights_for_predict():
    candidates = []

    # from previous cells
    for g in ["best_weights_n", "best_weights", "best_model_path", "best_model_x"]:
        if g in globals():
            try:
                p = Path(globals()[g])
                if p.exists() and p.name == "best.pt":
                    candidates.append(p)
            except Exception:
                pass

    # from current RUN_NAME (if available)
    run_name = globals().get("RUN_NAME")
    project_hint = globals().get("MLFLOW_PROJECT", "runs/colony_seg_mlflow_736")
    if run_name:
        for p in [
            Path(project_hint) / run_name / "weights" / "best.pt",
            Path("runs") / "segment" / Path(project_hint) / run_name / "weights" / "best.pt",
            Path("runs") / "segment" / "runs" / Path(project_hint).name / run_name / "weights" / "best.pt",
        ]:
            if p.exists():
                candidates.append(p)

    # latest n run directory
    for base in [
        Path("runs/segment/runs/colony_seg_mlflow_736"),
        Path("runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow_736"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
    ]:
        if not base.exists():
            continue
        for d in sorted(base.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if not d.is_dir() or d.name.endswith("_test"):
                continue
            if "yolo26n-seg" not in d.name.lower():
                continue
            p = d / "weights" / "best.pt"
            if p.exists():
                candidates.append(p)

    if not candidates:
        raise FileNotFoundError("Could not find best.pt for yolo26n-seg")

    # newest first
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def color_for_idx(i, n_total=1):
    # High-contrast deterministic instance colors.
    if n_total < 1:
        n_total = 1
    t = ((i * 37) % n_total) / max(1, n_total - 1)
    hue = (0.02 + 0.96 * t) % 1.0
    sat = 0.95
    val = 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
    return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


def render_masks_only(image_path, result):
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        return None, 0

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    overlay = rgb.copy()
    h, w = overlay.shape[:2]

    n_masks = 0
    if result.masks is not None and getattr(result.masks, "data", None) is not None:
        mask_data = result.masks.data
        if hasattr(mask_data, "detach"):
            mask_stack = mask_data.detach().cpu().numpy()
        else:
            mask_stack = np.asarray(mask_data)

        n_total = len(mask_stack)
        for k, mask_prob in enumerate(mask_stack):
            if mask_prob is None:
                continue

            mask_prob = np.asarray(mask_prob, dtype=np.float32)
            if mask_prob.ndim != 2:
                continue

            if mask_prob.shape != (h, w):
                mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

            # Soft mask edges for smoother visualization.
            mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
            mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
            if float(mask_alpha.max()) < 0.01:
                continue

            color = color_for_idx(k, n_total=n_total)
            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            binary = (mask_prob >= 0.5).astype(np.uint8)
            if binary.any():
                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                if contours:
                    cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

    elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
        # Fallback to polygon mode if data masks are unavailable.
        polys = result.masks.xy
        n_total = len(polys)
        for k, poly in enumerate(polys):
            if poly is None or len(poly) < 3:
                continue

            pts = np.round(poly).astype(np.int32)
            color = color_for_idx(k, n_total=n_total)

            mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
            n_masks += 1

    out = np.clip(overlay, 0, 255).astype(np.uint8)
    return out, n_masks


if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source folder not found: {SOURCE_DIR}")

images = sorted([p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
if not images:
    raise RuntimeError(f"No images found in: {SOURCE_DIR}")

weights_path = resolve_n_best_weights_for_predict()
print(f"Using weights: {weights_path}")
print(f"Source folder: {SOURCE_DIR.resolve()}")
print(f"Images count: {len(images)}")

pred_project = Path("runs/colony_seg_mlflow_736")
pred_name = f"n_predict_train_{datetime.now():%Y%m%d_%H%M%S}"

model = YOLO(str(weights_path))
results = model.predict(
    source=str(SOURCE_DIR),
    conf=PRED_CONF,
    iou=PRED_IOU,
    imgsz=PRED_IMGSZ,
    save=True,
    show_boxes=False,
    retina_masks=True,
    project=str(pred_project),
    name=pred_name,
    exist_ok=True,
    verbose=True,
)

# Resolve real save directory from Ultralytics output (robust to internal path prefixes)
pred_dir = None
if results:
    try:
        sd = getattr(results[0], "save_dir", None)
        if sd:
            pred_dir = Path(sd)
    except Exception:
        pass

if pred_dir is None or not pred_dir.exists():
    # fallback search by folder name
    matches = sorted(Path("runs").rglob(pred_name), key=lambda x: x.stat().st_mtime, reverse=True)
    if matches:
        pred_dir = matches[0]

if pred_dir is None or not pred_dir.exists():
    raise FileNotFoundError(f"Prediction output folder not found for: {pred_name}")

print(f"Saved predictions to: {pred_dir.resolve()}")

# Preview ONLY requested files with custom per-instance colors
preview_paths = [SOURCE_DIR / fn for fn in TARGET_PREVIEW_FILES]
missing = [str(p) for p in preview_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing preview files:\n" + "\n".join(missing))

print("Preview files:")
for p in preview_paths:
    print(f"- {p}")

preview_results = model.predict(
    source=[str(p) for p in preview_paths],
    conf=PRED_CONF,
    iou=PRED_IOU,
    imgsz=PRED_IMGSZ,
    save=False,
    show_boxes=False,
    retina_masks=True,
    verbose=False,
)

cols = 2
rows = int(np.ceil(len(preview_paths) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.8 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis("off")

for ax, img_path, res in zip(axes, preview_paths, preview_results):
    out, n_masks = render_masks_only(img_path, res)
    if out is None:
        ax.set_title(f"{img_path.name} (read error)", fontsize=9)
        continue
    ax.imshow(out)
    ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

# Save this custom preview panel
custom_png = pred_dir / "n_custom_preview_4363_4615_4672_6137_masks_only.png"
fig.savefig(custom_png, dpi=180, bbox_inches="tight")
print(f"Saved custom preview: {custom_png.resolve()}")



In [ ]:
# Predict on full cropped_736/images/train using latest yolo26x-seg best weights
from pathlib import Path
from datetime import datetime
import colorsys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SOURCE_DIR = Path("cropped_736/images/train")
TARGET_PREVIEW_FILES = [
    "1127763_IMG_4363.jpg",
    "1127763_IMG_4615.jpg",
    "1127763_IMG_4672.jpg",
    "1127763_IMG_6137.jpg",
]
PRED_CONF = 0.25
PRED_IOU = 0.6
PRED_IMGSZ = 736
MASK_ALPHA = 0.55
CONTOUR_THICKNESS = 2
SMOOTH_SIGMA = 0.9


def resolve_x_best_weights_for_predict():
    candidates = []

    # from previous cells
    for g in ["best_weights_x", "best_weights", "best_model_path", "best_model_x"]:
        if g in globals():
            try:
                p = Path(globals()[g])
                if p.exists() and p.name == "best.pt":
                    candidates.append(p)
            except Exception:
                pass

    # from current RUN_NAME (if available)
    run_name = globals().get("RUN_NAME")
    project_hint = globals().get("MLFLOW_PROJECT", "runs/colony_seg_mlflow_736")
    if run_name and "yolo26x-seg" in str(run_name).lower():
        for p in [
            Path(project_hint) / run_name / "weights" / "best.pt",
            Path("runs") / "segment" / Path(project_hint) / run_name / "weights" / "best.pt",
            Path("runs") / "segment" / "runs" / Path(project_hint).name / run_name / "weights" / "best.pt",
        ]:
            if p.exists():
                candidates.append(p)

    # latest x run directory
    for base in [
        Path("runs/segment/runs/colony_seg_mlflow_736"),
        Path("runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow_736"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
    ]:
        if not base.exists():
            continue
        for d in sorted(base.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if not d.is_dir() or d.name.endswith("_test"):
                continue
            if "yolo26x-seg" not in d.name.lower():
                continue
            p = d / "weights" / "best.pt"
            if p.exists():
                candidates.append(p)

    if not candidates:
        raise FileNotFoundError("Could not find best.pt for yolo26x-seg")

    # newest first
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def color_for_idx(i, n_total=1):
    # High-contrast deterministic instance colors.
    if n_total < 1:
        n_total = 1
    t = ((i * 37) % n_total) / max(1, n_total - 1)
    hue = (0.02 + 0.96 * t) % 1.0
    sat = 0.95
    val = 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
    return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


def render_masks_only(image_path, result):
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        return None, 0

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    overlay = rgb.copy()
    h, w = overlay.shape[:2]

    n_masks = 0
    if result.masks is not None and getattr(result.masks, "data", None) is not None:
        mask_data = result.masks.data
        if hasattr(mask_data, "detach"):
            mask_stack = mask_data.detach().cpu().numpy()
        else:
            mask_stack = np.asarray(mask_data)

        n_total = len(mask_stack)
        for k, mask_prob in enumerate(mask_stack):
            if mask_prob is None:
                continue

            mask_prob = np.asarray(mask_prob, dtype=np.float32)
            if mask_prob.ndim != 2:
                continue

            if mask_prob.shape != (h, w):
                mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

            # Soft mask edges for smoother visualization.
            mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
            mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
            if float(mask_alpha.max()) < 0.01:
                continue

            color = color_for_idx(k, n_total=n_total)
            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            binary = (mask_prob >= 0.5).astype(np.uint8)
            if binary.any():
                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                if contours:
                    cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

    elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
        # Fallback to polygon mode if data masks are unavailable.
        polys = result.masks.xy
        n_total = len(polys)
        for k, poly in enumerate(polys):
            if poly is None or len(poly) < 3:
                continue

            pts = np.round(poly).astype(np.int32)
            color = color_for_idx(k, n_total=n_total)

            mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
            n_masks += 1

    out = np.clip(overlay, 0, 255).astype(np.uint8)
    return out, n_masks


if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source folder not found: {SOURCE_DIR}")

images = sorted([p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
if not images:
    raise RuntimeError(f"No images found in: {SOURCE_DIR}")

weights_path = resolve_x_best_weights_for_predict()
print(f"Using weights: {weights_path}")
print(f"Source folder: {SOURCE_DIR.resolve()}")
print(f"Images count: {len(images)}")

pred_project = Path("runs/colony_seg_mlflow_736")
pred_name = f"x_predict_train_{datetime.now():%Y%m%d_%H%M%S}"

model = YOLO(str(weights_path))
results = model.predict(
    source=str(SOURCE_DIR),
    conf=PRED_CONF,
    iou=PRED_IOU,
    imgsz=PRED_IMGSZ,
    save=True,
    show_boxes=False,
    retina_masks=True,
    project=str(pred_project),
    name=pred_name,
    exist_ok=True,
    verbose=True,
)

# Resolve real save directory from Ultralytics output (robust to internal path prefixes)
pred_dir = None
if results:
    try:
        sd = getattr(results[0], "save_dir", None)
        if sd:
            pred_dir = Path(sd)
    except Exception:
        pass

if pred_dir is None or not pred_dir.exists():
    matches = sorted(Path("runs").rglob(pred_name), key=lambda x: x.stat().st_mtime, reverse=True)
    if matches:
        pred_dir = matches[0]

if pred_dir is None or not pred_dir.exists():
    raise FileNotFoundError(f"Prediction output folder not found for: {pred_name}")

print(f"Saved predictions to: {pred_dir.resolve()}")

# Preview ONLY requested files with custom per-instance colors
preview_paths = [SOURCE_DIR / fn for fn in TARGET_PREVIEW_FILES]
missing = [str(p) for p in preview_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing preview files\n" + "\n".join(missing))

print("Preview files:")
for p in preview_paths:
    print(f"- {p}")

preview_results = model.predict(
    source=[str(p) for p in preview_paths],
    conf=PRED_CONF,
    iou=PRED_IOU,
    imgsz=PRED_IMGSZ,
    save=False,
    show_boxes=False,
    retina_masks=True,
    verbose=False,
)

cols = 2
rows = int(np.ceil(len(preview_paths) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.8 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis("off")

for ax, img_path, res in zip(axes, preview_paths, preview_results):
    out, n_masks = render_masks_only(img_path, res)
    if out is None:
        ax.set_title(f"{img_path.name} (read error)", fontsize=9)
        continue
    ax.imshow(out)
    ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

# Save this custom preview panel
custom_png = pred_dir / "x_custom_preview_4363_4615_4672_6137_masks_only.png"
fig.savefig(custom_png, dpi=180, bbox_inches="tight")
print(f"Saved custom preview: {custom_png.resolve()}")



In [ ]:
# Compare mask metrics for yolo26n-seg vs yolo26x-seg on selected images
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from ultralytics import YOLO

COMPARE_IMAGE_FILES = [
    "1127763_IMG_4363.jpg",
    "1127763_IMG_4615.jpg",
    "1127763_IMG_4672.jpg",
    "1127763_IMG_6137.jpg",
]

PRED_CONF = 0.25
PRED_IOU = 0.6
PRED_IMGSZ = 736


def resolve_best_weights_for_model(model_tag: str):
    candidates = []

    # from globals (if previous cells ran)
    globals_map = {
        "yolo26n-seg": ["best_weights_n", "best_weights", "best_model_path"],
        "yolo26x-seg": ["best_weights_x", "best_weights", "best_model_path", "best_model_x"],
    }
    for g in globals_map.get(model_tag, []):
        if g in globals():
            try:
                p = Path(globals()[g])
                if p.exists() and p.name == "best.pt":
                    candidates.append(p)
            except Exception:
                pass

    # scan run dirs
    bases = [
        Path("runs/segment/runs/colony_seg_mlflow_736"),
        Path("runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow_736"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
    ]
    for base in bases:
        if not base.exists():
            continue
        for d in sorted(base.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if not d.is_dir() or d.name.endswith("_test"):
                continue
            if model_tag not in d.name.lower():
                continue
            p = d / "weights" / "best.pt"
            if p.exists():
                candidates.append(p)

    if not candidates:
        raise FileNotFoundError(f"Could not find best.pt for {model_tag}")

    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def read_yolo_seg_polygons(label_path: Path):
    anns = []
    if not label_path.exists():
        return anns

    txt = label_path.read_text(encoding="utf-8", errors="ignore")
    if not txt.strip():
        return anns

    txt = txt.replace("\r", "\n")
    for raw in txt.splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.replace(",", " ").split()
        if len(parts) < 7:
            continue
        vals = []
        for token in parts[1:]:
            try:
                vals.append(float(token))
            except Exception:
                pass
        if len(vals) < 6:
            continue
        if len(vals) % 2 == 1:
            vals = vals[:-1]
        if len(vals) < 6:
            continue
        pts = np.array(vals, dtype=np.float32).reshape(-1, 2)
        pts = np.clip(pts, 0.0, 1.0)
        if pts.shape[0] >= 3:
            anns.append(pts)
    return anns


def polygons_to_mask(polygons_norm, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)
    for pts in polygons_norm:
        arr = np.asarray(pts, dtype=np.float32)
        if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
            continue
        px = np.empty_like(arr)
        px[:, 0] = np.clip(arr[:, 0] * (w - 1), 0, w - 1)
        px[:, 1] = np.clip(arr[:, 1] * (h - 1), 0, h - 1)
        cv2.fillPoly(mask, [np.round(px).astype(np.int32)], 1)
    return mask


def predict_mask_union(result, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)
    pred_count = 0

    if result is None or result.masks is None:
        return mask, pred_count

    data = getattr(result.masks, "data", None)
    if data is not None:
        arr = data.detach().cpu().numpy() if hasattr(data, "detach") else np.asarray(data)
        pred_count = int(arr.shape[0]) if arr.ndim >= 3 else 0
        if pred_count == 0:
            return mask, pred_count
        if arr.shape[-2:] != (h, w):
            resized = []
            for m in arr:
                resized.append(cv2.resize(m.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR))
            arr = np.stack(resized, axis=0)
        bin_stack = (arr > 0.5).astype(np.uint8)
        mask = (bin_stack.max(axis=0) > 0).astype(np.uint8)
        return mask, pred_count

    polys = getattr(result.masks, "xy", None)
    if polys is not None:
        pred_count = int(len(polys))
        for poly in polys:
            if poly is None or len(poly) < 3:
                continue
            pts = np.asarray(poly, dtype=np.float32)
            if pts.ndim != 2 or pts.shape[1] != 2:
                continue
            pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
            pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)
            cv2.fillPoly(mask, [np.round(pts).astype(np.int32)], 1)

    return mask, pred_count


def binary_metrics(pred_mask, gt_mask):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    inter = np.logical_and(pred, gt).sum(dtype=np.int64)
    pred_sum = pred.sum(dtype=np.int64)
    gt_sum = gt.sum(dtype=np.int64)
    union = np.logical_or(pred, gt).sum(dtype=np.int64)

    dice = (2.0 * inter / (pred_sum + gt_sum)) if (pred_sum + gt_sum) > 0 else 1.0
    iou = (inter / union) if union > 0 else 1.0
    precision = (inter / pred_sum) if pred_sum > 0 else (1.0 if gt_sum == 0 else 0.0)
    recall = (inter / gt_sum) if gt_sum > 0 else (1.0 if pred_sum == 0 else 0.0)

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision_px": float(precision),
        "recall_px": float(recall),
        "gt_area_px": int(gt_sum),
        "pred_area_px": int(pred_sum),
    }


# Resolve data dirs
source_dir = Path("cropped_736/images/train")
if not source_dir.exists() and "SOURCE_DIR" in globals():
    source_dir = Path(SOURCE_DIR)

if not source_dir.exists():
    raise FileNotFoundError(f"Image dir not found: {source_dir}")

if source_dir.parent.name == "images":
    labels_dir = source_dir.parent.parent / "labels" / source_dir.name
else:
    labels_dir = Path("cropped_736/labels/train")

if not labels_dir.exists():
    raise FileNotFoundError(f"Labels dir not found: {labels_dir}")

# Resolve models
w_n = resolve_best_weights_for_model("yolo26n-seg")
w_x = resolve_best_weights_for_model("yolo26x-seg")

print("Using n weights:", w_n)
print("Using x weights:", w_x)
print("Images dir:", source_dir.resolve())
print("Labels dir:", labels_dir.resolve())

model_n = YOLO(str(w_n))
model_x = YOLO(str(w_x))

rows = []
for fn in COMPARE_IMAGE_FILES:
    img_path = source_dir / fn
    lbl_path = labels_dir / f"{img_path.stem}.txt"

    if not img_path.exists():
        rows.append({"model": "n", "image": fn, "error": "image_not_found"})
        rows.append({"model": "x", "image": fn, "error": "image_not_found"})
        continue

    bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if bgr is None:
        rows.append({"model": "n", "image": fn, "error": "image_read_error"})
        rows.append({"model": "x", "image": fn, "error": "image_read_error"})
        continue

    h, w = bgr.shape[:2]
    gt_polys = read_yolo_seg_polygons(lbl_path)
    gt_mask = polygons_to_mask(gt_polys, h, w)
    gt_count = len(gt_polys)

    res_n = model_n.predict(source=[str(img_path)], conf=PRED_CONF, iou=PRED_IOU, imgsz=PRED_IMGSZ, retina_masks=True, save=False, verbose=False)[0]
    res_x = model_x.predict(source=[str(img_path)], conf=PRED_CONF, iou=PRED_IOU, imgsz=PRED_IMGSZ, retina_masks=True, save=False, verbose=False)[0]

    pred_n, pred_count_n = predict_mask_union(res_n, h, w)
    pred_x, pred_count_x = predict_mask_union(res_x, h, w)

    m_n = binary_metrics(pred_n, gt_mask)
    m_x = binary_metrics(pred_x, gt_mask)

    rows.append({
        "model": "n",
        "image": fn,
        **m_n,
        "gt_instances": int(gt_count),
        "pred_instances": int(pred_count_n),
        "count_ae": float(abs(pred_count_n - gt_count)),
        "count_se": float((pred_count_n - gt_count) ** 2),
    })
    rows.append({
        "model": "x",
        "image": fn,
        **m_x,
        "gt_instances": int(gt_count),
        "pred_instances": int(pred_count_x),
        "count_ae": float(abs(pred_count_x - gt_count)),
        "count_se": float((pred_count_x - gt_count) ** 2),
    })

per_image_df = pd.DataFrame(rows)

num_cols = ["dice", "iou", "precision_px", "recall_px", "count_ae", "count_se", "gt_area_px", "pred_area_px", "gt_instances", "pred_instances"]
valid = per_image_df.copy()
for c in num_cols:
    if c in valid.columns:
        valid[c] = pd.to_numeric(valid[c], errors="coerce")

summary = valid.groupby("model", as_index=False).agg({
    "dice": "mean",
    "iou": "mean",
    "precision_px": "mean",
    "recall_px": "mean",
    "count_ae": "mean",
    "count_se": "mean",
    "gt_instances": "mean",
    "pred_instances": "mean",
})
if "count_se" in summary.columns:
    summary["count_rmse"] = np.sqrt(summary["count_se"])

try:
    display(per_image_df)
    display(summary.sort_values("dice", ascending=False))
except Exception:
    print(per_image_df)
    print(summary.sort_values("dice", ascending=False))

out_dir = Path("runs/segment/runs/colony_seg_mlflow_736")
out_dir.mkdir(parents=True, exist_ok=True)
per_img_csv = out_dir / "mask_compare_n_vs_x_per_image.csv"
summary_csv = out_dir / "mask_compare_n_vs_x_summary.csv"
per_image_df.to_csv(per_img_csv, index=False)
summary.to_csv(summary_csv, index=False)

print("Saved:", per_img_csv.resolve())
print("Saved:", summary_csv.resolve())

